# Is gas surface density a better tracer of the quenching *mode* than the quenching *time*?

**Scope.** A focused, hypothesis-driven re-run of the residual-dust pipeline across **multiple
selection redshifts** (z = 0.0, 0.4, 0.7, 0.9, 1.0, 4.0). For galaxies selected at each epoch
(realistically populated in stars *and* gas) we track histories back to formation, define the
critical points (formation, sSFR peak, AGN activation, SFT, QT, post-quench, gas_min), check the
histories and halo tracking are well-behaved, and split by **AGN–ISM coupling** (no-AGN / weak /
strong).

**Question.** Quenching time (τ_q; fast/slow) is a degenerate tracer of *how* a galaxy quenched.
Is the directly-measured **gas surface density** a better tracer of the coupling strength and of
the dust outcome at gas_min?

**End goal.** Do galaxies that are dustier at gas_min have a characteristic quenching mode
(fast/slow, weak/strong)? Is the post-gas_min sSFR rebound the norm for dusty galaxies, and does it
track τ_q or coupling? Operationalised in the Σ_H2–compactness(/age) plane (Part 6h).

**This notebook is self-contained and anchor-parameterised; it reuses the proven machinery of
`residual_dust_quenching.ipynb` (it does not modify it).** Heavy products are gated behind
`BUILD_*` flags and built on the cluster; with the gates off the notebook only loads + analyses.
All statistics use `scipy.stats` (the cluster `pd39` kernel has no statsmodels).

# Part 0 — Setup & configuration

In [ ]:
import os
import gc
import numpy as np
import h5py
import matplotlib.pyplot as plt

from astropy.io import fits
from astropy import units as u
from astropy import constants as const
from astropy.cosmology import Planck15 as COSMO   # matches the repo's quenching batch

# scipy.stats only — NO statsmodels on the cluster kernel (p-values would silently be NaN)
from scipy.stats import spearmanr, pearsonr, ks_2samp, fisher_exact, mannwhitneyu

from simbanator.io.simba import Simulation
from simbanator.analysis import HDF5BuildHistory, caesar_read_progen
from simbanator.analysis.quenching import find_quenching_times

# ── simulation / tracking config (identical to residual_dust_quenching.ipynb §0) ──
SIM_NAME    = "cis100"
BOX         = 100               # Mpc/h (sanity only)
START_SNAP  = 150               # z~0 anchor for the z=0 tracks
END_SNAP    = 44                # z~4.8

MASS_FLOOR      = 9.5
MASS_BIN_EDGES  = [9.5, 10.5, 11.0, np.inf]
MASS_BIN_LABELS = ["9.5-10.5", "10.5-11", ">=11"]
NBINS           = len(MASS_BIN_LABELS)
HMASS_BIN_EDGES  = [11.5, 12.0, 13.0, 14.5, np.inf]
HMASS_BIN_LABELS = ["11.5-12", "12-13", "13-14.5", ">=14.5"]
PASSIVE_FACTOR  = 0.2           # passive if sSFR < PASSIVE_FACTOR / t_H  (== QT threshold)
TAU_SPLIT_LOG   = -1.25         # fast/slow cut on log10(tau_q / t_H(z_QT))

# ── selection: "realistically populated" in BOTH stars and gas ──
NSTAR_MIN = 20                  # min star particles at the anchor epoch  (NEW vs §0)
NGAS_MIN  = 10                  # min gas particles  (raised from 5 for profile reliability)

# ── AGN / coupling constants (residual_dust §4b, §8j) ──
EPS_R          = 0.1            # radiative/feedback efficiency for the AGN energy proxy
JET_LOGMBH     = 7.5            # jet mode: log10(M_BH/Msun) > 7.5 ...
JET_FEDD       = 0.2           #           ... AND f_Edd < 0.2
XRAY_FEDD_MAX  = 0.02          # X-ray (full-velocity jet) coupling: f_Edd <= 0.02 (Dave+19)
XRAY_FGAS_MAX  = 0.2           # ... AND f_gas = Mgas/M* < 0.2
c2_erg_per_Msun = (const.c ** 2 * u.Msun).to("erg").value
GYR = 1e9                       # yr per Gyr (event-stack clock unit)

# fast/slow styling (group A = fast, group B = slow) — the engines key off these
COL_A, COL_B, LBL_A, LBL_B = "C3", "C0", "fast", "slow"

# ── the redshift anchors this whole notebook keys off ──
ANCHOR_REDSHIFTS = [0.0, 0.4, 0.7, 0.9, 1.0, 4.0]
TRACK_AGE_FRAC   = 0.09         # track back to ~this fraction of the cosmic age at selection
ANCHOR_END_OVERRIDE = {}
CORRUPT_SNAPS = set()

# ── heavy-build gates (set True on the cluster, then reuse cached products) ──
BUILD_MULTI_Z      = False      # per-anchor progenitor FITS + property history HDF5
BUILD_BH           = False      # per-anchor BH accretion / AGN-energy history
BUILD_PROFILE_PLAN = False      # write per-anchor dust_profile_plan_<tag>.hdf5 (then run SLURM)

# ── paths ──
sim     = Simulation(SIM_NAME)
OUT     = os.path.join(os.getcwd(), "output", SIM_NAME)
SFHDIR  = os.path.join(OUT, "caesar_sfh")
TABLEDIR = os.path.join(OUT, "tables")
PLOTDIR = os.path.join(OUT, "plots", "quench_mode_sigma")
os.makedirs(TABLEDIR, exist_ok=True)
os.makedirs(PLOTDIR, exist_ok=True)

PROG_FILE     = "progenitors_most_mass.fits"
HIST_FILE     = "history_passive_z0.hdf5"
HIST_PATH     = os.path.join(SFHDIR, HIST_FILE)
BH_HIST_PATH  = os.path.join(SFHDIR, "bh_history_passive_z0.hdf5")

def figpath(name):
    """Route a figure into a topical subfolder of PLOTDIR by filename prefix."""
    base = os.path.basename(name)
    sub = base.split("_")[0] if "_" in base else "misc"
    d = os.path.join(PLOTDIR, sub); os.makedirs(d, exist_ok=True)
    return os.path.join(d, base)

print("simulation :", SIM_NAME)
print("anchors    :", ANCHOR_REDSHIFTS)
print("selection  : passive + logM*>%.1f + nstar>=%d + ngas>=%d"
      % (MASS_FLOOR, NSTAR_MIN, NGAS_MIN))
print("z=0 t_H    : %.3f Gyr -> passive cut sSFR < %.2e /yr"
      % (COSMO.age(0).value, PASSIVE_FACTOR / (COSMO.age(0).value * 1e9)))

In [ ]:
# ── global plot style (verbatim from residual_dust_quenching.ipynb §0b) ───────
import functools
import matplotlib as mpl
from matplotlib.axes import Axes
from matplotlib.figure import Figure

PLOT_FIG_SCALE  = 1.45
PLOT_FONT_BASE  = 15
PLOT_FONT_FLOOR = 14

mpl.rcParams.update({
    "font.size":        PLOT_FONT_BASE,
    "axes.titlesize":   PLOT_FONT_BASE + 2,
    "axes.labelsize":   PLOT_FONT_BASE + 1,
    "xtick.labelsize":  PLOT_FONT_BASE,
    "ytick.labelsize":  PLOT_FONT_BASE,
    "legend.fontsize":  PLOT_FONT_BASE,
    "figure.titlesize": PLOT_FONT_BASE + 4,
    "axes.titleweight": "bold",
    "figure.dpi":       110,
    "lines.linewidth":  2.0,
})

def _bump(fs):
    if isinstance(fs, (int, float)) and not isinstance(fs, bool) and fs < PLOT_FONT_FLOOR:
        return PLOT_FONT_FLOOR
    return fs

def _patch_fontsize(cls, names):
    for nm in names:
        orig = getattr(cls, nm, None)
        if orig is None or getattr(orig, "_fontfloor", False):
            continue
        @functools.wraps(orig)
        def wrapper(*a, __orig=orig, **kw):
            if "fontsize" in kw: kw["fontsize"] = _bump(kw["fontsize"])
            if "size" in kw:     kw["size"]     = _bump(kw["size"])
            return __orig(*a, **kw)
        wrapper._fontfloor = True
        setattr(cls, nm, wrapper)

_patch_fontsize(Axes, ["set_title", "set_xlabel", "set_ylabel", "legend", "text", "annotate",
                       "set_xticklabels", "set_yticklabels"])
_patch_fontsize(Figure, ["suptitle", "legend"])
if not getattr(plt.figure, "_figscale", False):
    _orig_figure = plt.figure
    @functools.wraps(_orig_figure)
    def _figure(*a, **kw):
        base = kw.get("figsize") or mpl.rcParams["figure.figsize"]
        kw["figsize"] = (base[0] * PLOT_FIG_SCALE, base[1] * PLOT_FIG_SCALE)
        return _orig_figure(*a, **kw)
    _figure._figscale = True
    plt.figure = _figure
print(f"plot style: fonts>={PLOT_FONT_FLOOR}pt, figures x{PLOT_FIG_SCALE}")

# Part 1 — Multi-redshift anchors & per-epoch samples

Each anchor builds its **own** progenitor tracks + property history with that snapshot as row 0
(`BUILD_MULTI_Z`, cluster). The z=0 anchor reuses the existing `residual_dust_quenching.ipynb`
products. The sample at every epoch is one shared definition: **passive + massive + realistically
populated in stars and gas**.

In [ ]:
# property list to track (same families as residual_dust §1b; robust drop-and-retry on build)
PROPS = {
    "galaxy_data": [
        "masses.stellar", "sfr", "masses.gas", "masses.dust", "masses.H2", "masses.HI",
        "radii.stellar_half_mass", "radii.gas_half_mass",
        "radii.stellar_r20", "radii.stellar_r80", "radii.gas_r20", "radii.gas_r80",
        "pos", "ngas", "nstar", "ages.mass_weighted", "metallicities.mass_weighted",
        "metallicities.stellar",
    ],
    "halo_data": ["masses.total"],
}

def passive_sample_mask(P_, t_cosmic_yr_):
    """Passive + massive + realistically-populated-in-stars-and-gas, evaluated at row 0
    (the anchor epoch). Shared by every anchor. Returns (mask, dict of the individual cuts)."""
    mstar0 = P_["masses.stellar"][0]
    sfr0   = P_["sfr"][0]
    ngas0  = P_["ngas"][0]
    nstar0 = P_["nstar"][0] if "nstar" in P_ else np.full_like(mstar0, np.inf)
    ssfr0  = np.where(mstar0 > 0, sfr0 / mstar0, np.nan)
    cuts = {
        "passive":    ssfr0 < (PASSIVE_FACTOR / t_cosmic_yr_[0]),
        "massive":    np.log10(np.where(mstar0 > 0, mstar0, np.nan)) > MASS_FLOOR,
        "enoughgas":  ngas0  >= NGAS_MIN,
        "enoughstar": nstar0 >= NSTAR_MIN,
    }
    m = cuts["passive"] & cuts["massive"] & cuts["enoughgas"] & cuts["enoughstar"]
    return m, cuts

def _ztag(z):
    return ("z%g" % z).replace(".", "p")

def _prof_path(A):
    """Per-anchor dust/ISM particle-profile product (z=0 reuses the existing file)."""
    return (os.path.join(SFHDIR, "dust_profiles_allcrit.hdf5") if A["z_target"] == 0.0
            else os.path.join(SFHDIR, f"dust_profiles_allcrit_{A['tag']}.hdf5"))

In [ ]:
# ── anchor table: snapshot, track end, and every per-anchor product path ──
_sall, _zall = [], []
for _s in range(0, START_SNAP + 1):
    try:
        _zv = float(sim.get_z_from_snap(_s))
    except Exception:
        continue
    if np.isfinite(_zv) and _zv >= 0:
        _sall.append(_s); _zall.append(_zv)
_sall, _zall = np.asarray(_sall), np.asarray(_zall)
_aall = COSMO.age(_zall).value

ANCHORS = {}
for _zt in ANCHOR_REDSHIFTS:
    if _zt == 0.0:
        ANCHORS[_zt] = dict(z_target=0.0, tag=_ztag(0.0), snap=START_SNAP,
                            z=float(sim.get_z_from_snap(START_SNAP)), end_snap=END_SNAP,
                            prog_file=PROG_FILE, hist_path=HIST_PATH, bh_path=BH_HIST_PATH)
    else:
        _snap = int(_sall[np.argmin(np.abs(_zall - _zt))])
        _age_end = TRACK_AGE_FRAC * float(_aall[_sall == _snap][0])
        _end = int(ANCHOR_END_OVERRIDE.get(_zt, int(_sall[np.searchsorted(_aall, _age_end)])))
        _tag = _ztag(_zt)
        ANCHORS[_zt] = dict(z_target=_zt, tag=_tag, snap=_snap,
                            z=float(sim.get_z_from_snap(_snap)), end_snap=_end,
                            prog_file=f"progenitors_anchor_{_tag}.fits",
                            hist_path=os.path.join(SFHDIR, f"history_anchor_{_tag}.hdf5"),
                            bh_path=os.path.join(SFHDIR, f"bh_history_anchor_{_tag}.hdf5"))
    # each anchor gets its OWN plan dir so the SLURM job's profiles_part_*.hdf5 partials
    # (named by task id) never collide across anchors sharing SFHDIR.
    ANCHORS[_zt]["plan_path"] = os.path.join(SFHDIR, f"prof_{ANCHORS[_zt]['tag']}",
                                             f"dust_profile_plan_{ANCHORS[_zt]['tag']}.hdf5")
    ANCHORS[_zt]["prof_path"] = _prof_path(ANCHORS[_zt])

print(f"{'z_tgt':>6s} {'snap':>5s} {'z':>7s} {'end':>5s} {'hist':>6s} {'BH':>4s} {'prof':>5s}")
for _zt, A in ANCHORS.items():
    print(f"{_zt:6.1f} {A['snap']:5d} {A['z']:7.3f} {A['end_snap']:5d} "
          f"{'ok' if os.path.exists(A['hist_path']) else '--':>6s} "
          f"{'ok' if os.path.exists(A['bh_path']) else '--':>4s} "
          f"{'ok' if os.path.exists(A['prof_path']) else '--':>5s}")

In [ ]:
# ── 1z·build (GATED, cluster): per-anchor progenitor table + property history ──
# Verbatim port of residual_dust §1z·build; z=0 reuses the §1 products.
if BUILD_MULTI_Z:
    for _zt, A in ANCHORS.items():
        if _zt == 0.0 or os.path.exists(A["hist_path"]):
            print(f"[{A['tag']}] cached/reused -> {os.path.basename(A['hist_path'])}"); continue
        end = int(A["end_snap"])
        while end < A["snap"] and (end in CORRUPT_SNAPS or not os.path.exists(sim.get_caesar_file(end))):
            end += 1
        A["end_snap"] = end
        print(f"[{A['tag']}] anchor snap {A['snap']} (z={A['z']:.2f}) <- {end}: progenitor table ...")
        cs_a = sim.load_catalog(snap=A["snap"])
        caesar_read_progen([g.GroupID for g in cs_a.galaxies], A["prog_file"],
                           range(end, A["snap"] + 1), sim, output_dir=None)
        hist = HDF5BuildHistory(sim, cs_a, progfilename=A["prog_file"])
        with fits.open(hist.progen_file) as hdul:
            valid_ids = np.asarray(hdul[1].data["GroupID"])
            _tHa = COSMO.age(float(A["z"])).value * 1e9
            _gid = np.array([g.GroupID for g in cs_a.galaxies])
            _ms  = np.array([float(g.masses["stellar"]) for g in cs_a.galaxies])
            _sf  = np.array([float(g.sfr) for g in cs_a.galaxies])
            with np.errstate(all="ignore"):
                _ss = np.where(_ms > 0, _sf / _ms, np.nan)
                _ok = (np.log10(np.where(_ms > 0, _ms, np.nan)) >= MASS_FLOOR) & (_ss < PASSIVE_FACTOR / _tHa)
            _keep = {int(g) for g in _gid[_ok]}
            valid_ids = np.asarray([i for i in valid_ids if int(i) in _keep], dtype=valid_ids.dtype)
            print(f"  [pre-select] {len(valid_ids)}/{len(_gid)} massive+quenched at z={A['z']:.2f}")
        hist.get_history_indx(valid_ids, A["snap"], end)
        props_try = {k: list(v) for k, v in PROPS.items()}
        while True:
            try:
                hist.get_property_history(props_try, verbose=0); break
            except KeyError as e:
                msg = str(e); dropped = False
                for fam, plist in props_try.items():
                    for pr in list(plist):
                        if pr in msg or pr.split("/")[-1] in msg:
                            plist.remove(pr); print("  [drop]", pr); dropped = True
                if not dropped:
                    raise
        hist.save_history_to_hdf5(os.path.basename(A["hist_path"]))
        del cs_a, hist; gc.collect()
        print(f"[{A['tag']}] history -> {A['hist_path']}")
else:
    print("BUILD_MULTI_Z=False -> expecting per-anchor histories under", SFHDIR)

In [ ]:
# ── 1z·load — generic anchor loader (row 0 of every array = the anchor epoch) ──
def load_anchor_history(A):
    """Load one anchor's history -> dict(galaxy_ids, snaps_arr, redshift, t_cosmic_yr, P)."""
    H = {"P": {}}
    with h5py.File(A["hist_path"], "r") as f:
        H["galaxy_ids"] = f["metadata/galaxy_ids"][:]
        H["snaps_arr"]  = f["metadata/snapshots"][:]
        H["redshift"]   = f["redshift/Redshift"][:]
        f["properties"].visititems(
            lambda name, obj: H["P"].__setitem__(name, obj[:]) if isinstance(obj, h5py.Dataset) else None)
    H["t_cosmic_yr"] = COSMO.age(H["redshift"]).value * 1e9
    return H

def build_prog_index(A, galaxy_ids, snaps_arr):
    """(n_snap, n_gal) catalogue group-index matrix aligned to the anchor history rows.
    Needed only to BUILD the BH history and the profile plan (catalogue load)."""
    cs0 = sim.load_catalog(snap=A["snap"])
    hP = HDF5BuildHistory(sim, cs0, progfilename=A["prog_file"])
    hP.get_history_indx(galaxy_ids, int(np.max(snaps_arr)), int(np.min(snaps_arr)))
    M = np.vstack([hP.history_indx[str(s)] for s in snaps_arr])
    del cs0, hP; gc.collect()
    return M

# Part 2 — Machinery (anchor-parameterised functions)

Each function takes one anchor's arrays and returns its products, so the same proven logic runs at
every redshift. Bodies are ported from `residual_dust_quenching.ipynb` (§3 critical points,
§2·diag QC, §4b BH, §8j coupling, §5 profiles, §8e/§8g clock engine, §7 plotters).

In [ ]:
# ── §3 → build_records: critical points + fast/slow + mass bins (verbatim §3 logic) ──
def build_records(P, t_cosmic_yr, redshift, galaxy_ids, cols,
                  MASS_BIN_EDGES, HMASS_BIN_EDGES, PASSIVE_FACTOR, TAU_SPLIT_LOG, COSMO,
                  find_quenching_times):
    """Per-galaxy critical evolutionary points & fast/slow classification for ONE anchor.
    `cols` = column indices (into n_gal) of the selected sample. Adds a `formation` stage
    (earliest snapshot with M*>0) to the residual_dust STAGES list."""
    t_cosmic_yr = np.asarray(t_cosmic_yr, float); redshift = np.asarray(redshift, float)
    galaxy_ids = np.asarray(galaxy_ids); cols = np.asarray(cols, int)
    sample_gids = galaxy_ids[cols]
    STAGES = ["formation", "sf_peak", "ssfr_min", "sft", "qt", "post_quench",
              "dust_peak", "dust_min", "gas_min", "z0"]
    SMOOTH_W = 3

    def _nearest_row(t_target):
        if not np.isfinite(t_target):
            return -1
        return int(np.argmin(np.abs(t_cosmic_yr - t_target)))

    def _smooth(y, w=SMOOTH_W):
        n = len(y); h = w // 2; out = np.full(n, np.nan)
        for i in range(n):
            seg = y[max(0, i - h):min(n, i + h + 1)]; seg = seg[np.isfinite(seg)]
            if seg.size:
                out[i] = np.median(seg)
        return out

    def _ordered_positive(q, valid):
        d = np.where(valid, q, np.nan).astype(float)
        pos = np.isfinite(d) & (d > 0)
        if pos.sum() < 2:
            return None, None
        order = np.where(pos)[0]; order = order[np.argsort(t_cosmic_yr[order])]
        return order, _smooth(d[order])

    def _peak_time(q, valid):
        order, ds = _ordered_positive(q, valid)
        if order is None or not np.isfinite(ds).any():
            return np.nan
        return t_cosmic_yr[order[np.nanargmax(ds)]]

    def _trough_before_floor(q, valid, floor_frac=1e-6):
        order, ds = _ordered_positive(q, valid)
        if order is None or not np.isfinite(ds).any():
            return np.nan
        floor = floor_frac * np.nanmax(ds)
        below = np.isfinite(ds) & (ds <= floor)
        cut = int(np.argmax(below)) if below.any() else len(ds)
        cut = max(cut, 2); seg = ds[:cut]
        if not np.isfinite(seg).any():
            return np.nan
        return t_cosmic_yr[order[np.nanargmin(seg)]]

    def _formation_row(col):
        mstar = P["masses.stellar"][:, col]
        has = np.where(np.isfinite(mstar) & (mstar > 0))[0]
        return int(has.max()) if has.size else -1

    records = []
    for col, gid in zip(cols, sample_gids):
        mstar = P["masses.stellar"][:, col]; sfr = P["sfr"][:, col]
        dust = P["masses.dust"][:, col]; gas = P["masses.gas"][:, col]
        ssfr = np.where(mstar > 0, sfr / mstar, np.nan)
        valid = np.isfinite(ssfr) & (ssfr > 0) & np.isfinite(t_cosmic_yr)
        if valid.sum() < 5:
            continue
        t = t_cosmic_yr[valid]; s = ssfr[valid]
        o = np.argsort(t); t, s = t[o], s[o]
        tu, ui = np.unique(t, return_index=True); su = s[ui]
        if len(tu) < 5:
            continue
        qts, sfts, _, dbg = find_quenching_times(
            tu, su, galaxy_id=int(gid), plot=False, save_fits_path=None, return_debug=True)
        if len(qts) == 0:
            continue
        k = int(np.argmax(qts)); t_qt, t_sft = qts[k], sfts[k]
        t_postq = dbg["persistence_end_times"][k]
        kf = int(np.argmin(qts)); t_qt_first, t_sft_first = qts[kf], sfts[kf]
        t_sfpeak = _peak_time(ssfr, valid); t_ssfrmin = _trough_before_floor(ssfr, valid)
        t_dpeak = _peak_time(dust, valid); t_dmin = _trough_before_floor(dust, valid)
        t_gasmin = _trough_before_floor(gas, valid); t_z0 = t_cosmic_yr[0]
        row_form = _formation_row(col); t_form = t_cosmic_yr[row_form] if row_form >= 0 else np.nan
        t_times = {"formation": t_form, "sf_peak": t_sfpeak, "ssfr_min": t_ssfrmin,
                   "sft": t_sft, "qt": t_qt, "post_quench": t_postq, "dust_peak": t_dpeak,
                   "dust_min": t_dmin, "gas_min": t_gasmin, "z0": t_z0}
        rows = {st: _nearest_row(t_times[st]) for st in STAGES}
        rows["formation"] = row_form
        tau_q = t_qt - t_sft
        z_qt = float(np.interp(t_qt, t_cosmic_yr[::-1], redshift[::-1]))
        tH_qt = COSMO.age(z_qt).value * 1e9
        records.append(dict(
            gid=int(gid), col=int(col), t_formation=t_form,
            t_sf_peak=t_sfpeak, t_ssfr_min=t_ssfrmin, t_sft=t_sft, t_qt=t_qt,
            t_post_quench=t_postq, t_dust_peak=t_dpeak, t_dust_min=t_dmin,
            t_gas_min=t_gasmin, t_z0=t_z0,
            tau_q=tau_q, tau_q_over_tH=tau_q / tH_qt, z_qt=z_qt,
            t_qt_first=t_qt_first, t_sft_first=t_sft_first,
            **{f"row_{st}": rows[st] for st in STAGES}))

    n = len(records)
    cols_rec = np.array([r["col"] for r in records], int) if n else np.array([], int)
    gids_rec = np.array([r["gid"] for r in records]) if n else np.array([])
    tau_q = np.array([r["tau_q"] for r in records]) if n else np.array([])
    tau_n = np.array([r["tau_q_over_tH"] for r in records]) if n else np.array([])
    is_fast = np.log10(tau_n) < TAU_SPLIT_LOG if n else np.array([], bool)
    is_slow = (np.isfinite(tau_n) & ~is_fast) if n else np.array([], bool)
    if n:
        ms0 = P["masses.stellar"][0, cols_rec]; mh0 = P["masses.total"][0, cols_rec]
        logm_rec = np.log10(np.where(ms0 > 0, ms0, np.nan))
        logmh_rec = np.log10(np.where(mh0 > 0, mh0, np.nan))
    else:
        logm_rec = np.array([]); logmh_rec = np.array([])
    mbin = np.full(n, -1, int)
    for b in range(len(MASS_BIN_EDGES) - 1):
        mbin[(logm_rec >= MASS_BIN_EDGES[b]) & (logm_rec < MASS_BIN_EDGES[b + 1])] = b
    hmbin = np.full(n, -1, int)
    for b in range(len(HMASS_BIN_EDGES) - 1):
        hmbin[(logmh_rec >= HMASS_BIN_EDGES[b]) & (logmh_rec < HMASS_BIN_EDGES[b + 1])] = b
    return dict(records=records, STAGES=STAGES, tau_q=tau_q, tau_q_over_tH=tau_n,
                is_fast=is_fast, is_slow=is_slow, mbin=mbin, hmbin=hmbin,
                logm_rec=logm_rec, logmh_rec=logmh_rec, gids=gids_rec, cols=cols_rec)

In [ ]:
# ── §2·diag QC: well-behaved histories & halo tracking (per-galaxy boolean mask) ──
# Ported from residual_dust §2·diag2 (spatial continuity), §2·diag4 (halo-mass continuity),
# S2 (halo association). Thresholds copied verbatim. `redshift` passed directly (the anchor
# history carries it) instead of inverting COSMO.age.
def qc_anchor(P, t_cosmic_yr, redshift, snaps_arr, galaxy_ids, cols, BOX, tag):
    JUMP_FRAC, SPEED_HI = 0.05, 0.05      # §2·diag2: step>5% box; box-frac/Gyr implausible
    RATE_HI = 0.5                          # §2·diag4: dex/Gyr implausible
    BAD_FRAC, MIN_STEPS = 0.2, 3           # >20% over-rate steps & need >=3 steps to judge
    SHMR_HI = 0.2                          # S2: M*/Mhalo>0.2 suspicious
    cols = np.asarray(cols, int); ncol = cols.size
    t = np.asarray(t_cosmic_yr, float); dt = np.abs(np.diff(t)) / 1e9
    z = np.asarray(redshift, float)

    def _to_comoving(posc):
        fin = np.isfinite(posc)
        if not fin.any():
            return posc, 1.0
        def _ext(row):
            a = posc[row][np.isfinite(posc[row])]
            return float(np.nanmax(a) - np.nanmin(a)) if a.size else np.nan
        bs, be = _ext(0), _ext(-1)
        as_, ae = 1.0 / (1.0 + z[0]), 1.0 / (1.0 + z[-1])
        phys = (np.isfinite(bs) and np.isfinite(be)
                and abs(be / bs - ae / as_) < abs(be / bs - 1.0))
        if phys:
            out = posc * (1.0 + z)[:, None, None]
            L = bs / as_ if np.isfinite(bs) else np.nanpercentile(out[np.isfinite(out)], 99.99)
        else:
            out = posc
            L = bs if np.isfinite(bs) else np.nanpercentile(out[np.isfinite(out)], 99.99)
        return out, float(L)

    spatial_ok = np.ones(ncol, bool); n_broken = 0
    if "pos" in P:
        _pos = np.asarray(P["pos"], float)
        if _pos.ndim == 3 and _pos.shape[-1] == 3:
            posc, L = _to_comoving(_pos[:, cols, :])
            d = np.diff(posc, axis=0); d -= L * np.round(d / L)
            disp = np.sqrt(np.sum(d ** 2, axis=2)) / L
            with np.errstate(all="ignore"):
                speed = disp / np.where(dt > 0, dt, np.nan)[:, None]
                galbad = np.nansum(speed > SPEED_HI, axis=0) / np.maximum(np.sum(np.isfinite(speed), axis=0), 1)
            tracked = np.sum(np.isfinite(disp), axis=0) >= MIN_STEPS
            broken = tracked & (galbad > BAD_FRAC); n_broken = int(broken.sum()); spatial_ok = ~broken

    halo_ok = np.ones(ncol, bool); n_tele = 0
    if "masses.total" in P:
        _mh = np.asarray(P["masses.total"], float)[:, cols]
        with np.errstate(all="ignore"):
            lmh = np.log10(np.where(_mh > 0, _mh, np.nan))
            rate = np.abs(np.diff(lmh, axis=0)) / np.where(dt > 0, dt, np.nan)[:, None]
            halobad = np.nansum(rate > RATE_HI, axis=0) / np.maximum(np.sum(np.isfinite(rate), axis=0), 1)
        trk = np.sum(np.isfinite(np.diff(lmh, axis=0)), axis=0) >= MIN_STEPS
        susp = trk & (halobad > BAD_FRAC); n_tele = int(susp.sum()); halo_ok = ~susp

    _ms0 = np.asarray(P["masses.stellar"], float)[0, cols]
    _mh0 = np.asarray(P["masses.total"], float)[0, cols]
    with np.errstate(all="ignore"):
        shmr = np.where(_mh0 > 0, _ms0 / _mh0, np.nan)
    bad_ratio = np.isfinite(shmr) & (shmr > SHMR_HI)
    bad_order = np.isfinite(_ms0) & np.isfinite(_mh0) & (_mh0 < _ms0)
    no_halo = ~np.isfinite(_mh0) | (_mh0 <= 0)
    assoc_ok = ~(bad_ratio | bad_order | no_halo)
    median_shmr = float(np.nanmedian(shmr))

    well_behaved = spatial_ok & halo_ok & assoc_ok
    print(f"{tag}: well-behaved {int(well_behaved.sum())}/{ncol} | broken-link {n_broken} "
          f"| halo-jump {n_tele} | bad-assoc {int((bad_ratio | bad_order).sum())} "
          f"| no-halo {int(no_halo.sum())} | median M*/Mh {median_shmr:.4f}")
    return dict(well_behaved=well_behaved, n_broken_link=n_broken, n_halo_teleport=n_tele,
                n_bad_assoc=int((bad_ratio | bad_order).sum()), n_no_halo=int(no_halo.sum()),
                median_shmr=median_shmr)

In [ ]:
# ── §4b/§7j BH: per-anchor build + load + AGN metrics + AGN-activation time ──
BH_CANDIDATES = {"bh_mass": ["masses.bh", "masses.bh_mass", "bhmass"],
                 "bh_mdot": ["bhmdot", "bh_mdot"],
                 "bh_fedd": ["bh_fedd", "bhfedd", "fedd"]}

def _resolve_bh_path(f, cands):
    for c in cands:
        for p in (f"galaxy_data/dicts/{c}", f"galaxy_data/{c}"):
            if p in f:
                return p
    return None

def build_bh_for_anchor(A, galaxy_ids, snaps_arr, n_gal):
    """Verbatim port of residual_dust §1 BH build, per anchor -> A['bh_path']."""
    pidx = build_prog_index(A, galaxy_ids, snaps_arr)
    n_snap = len(snaps_arr)
    BH = {k: np.full((n_snap, n_gal), np.nan) for k in BH_CANDIDATES}
    for ri, snap in enumerate(snaps_arr):
        snap = int(snap)
        if snap in CORRUPT_SNAPS:
            continue
        try:
            with h5py.File(sim.get_caesar_file(snap), "r") as f:
                valid = np.isfinite(pidx[ri]); cv = np.where(valid)[0]
                vi = pidx[ri][valid].astype(int)
                for k, cands in BH_CANDIDATES.items():
                    p = _resolve_bh_path(f, cands)
                    if p is not None:
                        BH[k][ri, cv] = f[p][:][vi]
        except (OSError, KeyError) as e:
            print(f"  [skip] snap {snap}: {type(e).__name__}"); CORRUPT_SNAPS.add(snap)
    with h5py.File(A["bh_path"], "w") as f:
        for k, arr in BH.items():
            f.create_dataset(k, data=arr)
    print(f"[{A['tag']}] BH history -> {A['bh_path']}")
    return BH

def load_bh(bh_hist_path):
    with h5py.File(bh_hist_path, "r") as f:
        return {k: f[k][:] for k in f.keys()}

def bh_metrics(BH, records, t_cosmic_yr, EPS_R, c2_erg_per_Msun, JET_LOGMBH, JET_FEDD):
    def _sft_qt_rows(r):
        rs, rq = r["row_sft"], r["row_qt"]
        if rs < 0 or rq < 0:
            return np.array([], int)
        lo, hi = sorted((rs, rq)); return np.arange(lo, hi + 1)
    n = len(records); dMbh, Eagn, jetfrac, mdot_qt = (np.full(n, np.nan) for _ in range(4))
    for i, r in enumerate(records):
        col = r["col"]
        mdot_qt[i] = BH["bh_mdot"][r["row_qt"], col] if r["row_qt"] >= 0 else np.nan
        idx = _sft_qt_rows(r)
        if len(idx) < 2:
            continue
        t = t_cosmic_yr[idx]; o = np.argsort(t); idx = idx[o]; t = t[o]
        md = BH["bh_mdot"][idx, col]; mb = BH["bh_mass"][idx, col]; fe = BH["bh_fedd"][idx, col]
        good = np.isfinite(md) & np.isfinite(t)
        if good.sum() >= 2:
            Eagn[i] = EPS_R * np.trapz(md[good], t[good]) * c2_erg_per_Msun
        if np.isfinite(mb).sum() >= 2:
            mbv = mb[np.isfinite(mb)]; dMbh[i] = mbv[-1] - mbv[0]
        jet = np.isfinite(mb) & np.isfinite(fe) & (mb > 10 ** JET_LOGMBH) & (fe < JET_FEDD)
        vs = np.isfinite(mb) & np.isfinite(fe)
        jetfrac[i] = jet.sum() / vs.sum() if vs.sum() > 0 else np.nan
    return dict(Eagn=Eagn, dMbh=dMbh, jetfrac=jetfrac, mdot_qt=mdot_qt)

def agn_activation_time(BH, records, t_cosmic_yr, IGN_FRAC=0.5, IGN_FLOOR=1e-3, GYR=1e9):
    """t_AGN = first upward crossing of IGN_FRAC*(pre-quench peak BHAR) at/before QT (§8c)."""
    _ord = np.argsort(t_cosmic_yr); t_inc = t_cosmic_yr[_ord]
    def _track(arr2d, col):
        return arr2d[_ord, col].astype(float)
    t_agn = np.full(len(records), np.nan)
    for i, r in enumerate(records):
        col, tqt = r["col"], r["t_qt"]
        before = t_inc <= tqt + 0.05 * GYR
        md = _track(BH["bh_mdot"], col)
        m = before & np.isfinite(md) & (md > IGN_FLOOR)
        if m.sum() >= 2:
            thr = max(IGN_FRAC * np.nanmax(md[m]), IGN_FLOOR)
            cr = np.where(before & np.isfinite(md) & (md >= thr))[0]
            if len(cr):
                t_agn[i] = t_inc[cr[0]]
    return t_agn

In [ ]:
# ── §8j AGN–ISM coupling (verbatim): xstr_post strength split = SPLIT_XS ──
# Coupling strength is the mean of xcoup_hist over snapshots AFTER t_AGN (xstr_post), exactly as
# §8j. SPLIT_XS = top/bottom terciles of xstr_post ("strong"/"weak"); SPLIT_X = strict binary
# (xfrac_post>0). no_agn = ignited-but-never-coupled (xstr_post==0). An optional mass window
# (in_win) reproduces §8j's intermediate-mass restriction; default = whole sample.
def build_coupling(BH, P, records, t_cosmic_yr, t_agn,
                   JET_LOGMBH, JET_FEDD, XRAY_FEDD_MAX, XRAY_FGAS_MAX, in_win=None):
    _ord = np.argsort(t_cosmic_yr); t_inc = t_cosmic_yr[_ord]
    def _track(arr2d, col):
        return arr2d[_ord, col].astype(float)
    with np.errstate(all="ignore"):
        fgas_hist = np.where(P["masses.stellar"] > 0, P["masses.gas"] / P["masses.stellar"], np.nan)
        _bh_ok = np.isfinite(BH["bh_mass"]) & np.isfinite(BH["bh_fedd"])
        _mbh_ok = BH["bh_mass"] > 10 ** JET_LOGMBH
        wjet_hist = np.where(_bh_ok, np.where(_mbh_ok,
                             np.clip(np.log10(JET_FEDD / np.clip(BH["bh_fedd"], 1e-12, None)), 0.0, 1.0),
                             0.0), np.nan)
        xray_hist = np.where(_bh_ok & np.isfinite(fgas_hist),
                             (_mbh_ok & (BH["bh_fedd"] < XRAY_FEDD_MAX) & (fgas_hist < XRAY_FGAS_MAX)).astype(float),
                             np.nan)
        xcoup_hist = np.where(np.isfinite(wjet_hist) & np.isfinite(fgas_hist),
                              wjet_hist * (fgas_hist < XRAY_FGAS_MAX).astype(float), np.nan)
    n = len(records)
    xfrac_post = np.full(n, np.nan); xstr_post = np.full(n, np.nan)   # post-t_AGN strict frac / strength
    for i, r in enumerate(records):
        if not np.isfinite(t_agn[i]):
            continue
        post = t_inc >= t_agn[i]
        xs = _track(xray_hist, r["col"]); ok = post & np.isfinite(xs)
        if ok.sum() >= 2:
            xfrac_post[i] = np.nanmean(xs[ok])
        cs = _track(xcoup_hist, r["col"]); ok = post & np.isfinite(cs)
        if ok.sum() >= 2:
            xstr_post[i] = np.nanmean(cs[ok])
    base = np.ones(n, bool) if in_win is None else np.asarray(in_win, bool)
    bx = base & np.isfinite(xstr_post)
    lo_q = hi_q = np.nan; strong = np.zeros(n, bool); weak = np.zeros(n, bool)
    if bx.sum() >= 3:
        lo_q, hi_q = np.nanquantile(xstr_post[bx], [1.0 / 3.0, 2.0 / 3.0])
        strong = bx & (xstr_post >= hi_q); weak = bx & (xstr_post <= lo_q)   # §8j SPLIT_XS terciles
    no_agn = base & np.isfinite(xstr_post) & (xstr_post == 0)                 # ignited, never coupled
    SPLIT_X  = [(base & np.isfinite(xfrac_post) & (xfrac_post > 0),  "C3", "ISM-coupled"),
                (base & np.isfinite(xfrac_post) & (xfrac_post == 0), "C0", "never coupled")]
    SPLIT_XS = [(strong, "C3", f"strong coupling (>={hi_q:.2f})"),
                (weak,   "C0", f"weak coupling (<={lo_q:.2f})")]
    return dict(fgas_hist=fgas_hist, wjet_hist=wjet_hist, xray_hist=xray_hist, xcoup_hist=xcoup_hist,
                xfrac_post=xfrac_post, xstr_post=xstr_post,
                no_agn=no_agn, weak=weak, strong=strong, tercile=(lo_q, hi_q),
                SPLIT_X=SPLIT_X, SPLIT_XS=SPLIT_XS)

In [ ]:
# ── §5 particle profiles: per-anchor plan, loader, and Sigma scalars ──
def build_profile_plan(records, STAGES, snaps_arr, _PROG_INDEX, plan_path, sim_name,
                       rmax_kpc=50.0, nbins_r=25, store_sigma=1):
    """Write the per-anchor SLURM extraction plan (residual_dust §5a)."""
    _ri, _si, _gx, _snap = [], [], [], []
    for ri, r in enumerate(records):
        for si, st in enumerate(STAGES):
            row = r[f"row_{st}"]
            if row < 0:
                continue
            gx = _PROG_INDEX[row, r["col"]]
            if not np.isfinite(gx):
                continue
            _ri.append(ri); _si.append(si); _gx.append(int(gx)); _snap.append(int(snaps_arr[row]))
    os.makedirs(os.path.dirname(plan_path), exist_ok=True)
    with h5py.File(plan_path, "w") as out:
        out.attrs["nrec"] = len(records); out.attrs["nst"] = len(STAGES)
        out.attrs["sim_name"] = sim_name; out.attrs["rmax_kpc"] = rmax_kpc
        out.attrs["nbins_r"] = nbins_r; out.attrs["store_sigma"] = int(store_sigma)
        out.attrs["stages"] = ",".join(STAGES)
        out.create_dataset("entry_ri", data=np.array(_ri, np.int32))
        out.create_dataset("entry_si", data=np.array(_si, np.int32))
        out.create_dataset("entry_gx", data=np.array(_gx, np.int64))
        out.create_dataset("entry_snap", data=np.array(_snap, np.int32))
        out.create_dataset("gid", data=np.array([r["gid"] for r in records], np.int64))
    print(f"wrote plan: {len(_ri)} (galaxy,stage) entries -> {plan_path}")
    return plan_path

def _align_records_by_gid(arrays, cache_gid, cur_gid, axis=0, name=""):
    cache_gid = np.asarray(cache_gid).astype(np.int64); cur_gid = np.asarray(cur_gid).astype(np.int64)
    if len(cache_gid) == len(cur_gid) and np.array_equal(cache_gid, cur_gid):
        return arrays, len(cur_gid)
    pos = {int(g): j for j, g in enumerate(cache_gid)}
    sel = np.array([pos.get(int(g), -1) for g in cur_gid]); found = sel >= 0
    out = {}
    for k, a in arrays.items():
        a = np.asarray(a)
        if a.ndim <= axis or a.shape[axis] != len(cache_gid):
            out[k] = a; continue
        nshape = list(a.shape); nshape[axis] = len(cur_gid)
        b = np.full(nshape, np.nan, float); idx = np.where(found)[0]
        sl = [slice(None)] * b.ndim; sl[axis] = idx
        b[tuple(sl)] = np.take(a, sel[found], axis=axis); out[k] = b
    print(f"[warn] {name}: cache {len(cache_gid)} vs sample {len(cur_gid)} "
          f"({int(found.sum())} matched, {int((~found).sum())} -> NaN).")
    return out, int(found.sum())

def load_profiles(dustprof_path, records):
    DP_COMPS = ("dust", "HI", "H2", "star")
    if not os.path.exists(dustprof_path):
        return {"DP": None, "DP_OK": False, "DP_STAGES": None, "DP_RMID": None,
                "DP_COMPS": DP_COMPS, "DP_RATIO": None, "DP_C2050": {}, "DP_C2080": {},
                "DP_HAS_CONC": False}
    with h5py.File(dustprof_path, "r") as f:
        DP_STAGES = list(f.attrs["stages"].split(",")); DP = {k: f[k][:] for k in f.keys()}
    _cur = np.array([r["gid"] for r in records])
    if "gid" in DP:
        DP, _ = _align_records_by_gid(DP, DP["gid"], _cur, name="dust profiles"); DP["gid"] = _cur
    with np.errstate(all="ignore"):
        DP_RATIO = np.where(DP["R50_star"] > 0, DP["R50_dust"] / DP["R50_star"], np.nan)
        DP_HAS_CONC = all(f"R20_{c}" in DP and f"R80_{c}" in DP for c in DP_COMPS)
        DP_C2050, DP_C2080 = {}, {}
        for c in DP_COMPS:
            if DP_HAS_CONC:
                DP_C2050[c] = np.where(DP[f"R50_{c}"] > 0, DP[f"R20_{c}"] / DP[f"R50_{c}"], np.nan)
                DP_C2080[c] = np.where(DP[f"R80_{c}"] > 0, DP[f"R20_{c}"] / DP[f"R80_{c}"], np.nan)
            else:
                DP_C2050[c] = np.full_like(DP["R50_dust"], np.nan)
                DP_C2080[c] = np.full_like(DP["R50_dust"], np.nan)
    print(f"loaded {os.path.basename(dustprof_path)}: stages={DP_STAGES}; "
          f"conc={'present' if DP_HAS_CONC else 'absent'}")
    return {"DP": DP, "DP_OK": True, "DP_STAGES": DP_STAGES, "DP_RMID": DP["rmid"],
            "DP_COMPS": DP_COMPS, "DP_RATIO": DP_RATIO, "DP_C2050": DP_C2050,
            "DP_C2080": DP_C2080, "DP_HAS_CONC": DP_HAS_CONC}

def sigma_scalars(DP):
    """Mean surface density interior to R50_star (per record, per stage). The §5 product has no
    total-gas Sigma, so Sigma_gas_eff = HI + H2."""
    rmid = np.asarray(DP["rmid"])
    sd, sHI, sH2 = (np.asarray(DP[k]) for k in ("sigma_dust", "sigma_HI", "sigma_H2"))
    R50 = np.asarray(DP["R50_star"]); nrec, nst = R50.shape
    out = {k: np.full((nrec, nst), np.nan) for k in ("Sigma_dust_eff", "Sigma_HI_eff", "Sigma_H2_eff", "Sigma_gas_eff")}
    with np.errstate(all="ignore"):
        for ri in range(nrec):
            for si in range(nst):
                r50 = R50[ri, si]
                if not np.isfinite(r50) or r50 <= 0:
                    continue
                inb = rmid <= r50
                if not np.any(inb):
                    continue
                out["Sigma_dust_eff"][ri, si] = np.nanmean(sd[ri, si, inb])
                out["Sigma_HI_eff"][ri, si] = np.nanmean(sHI[ri, si, inb])
                out["Sigma_H2_eff"][ri, si] = np.nanmean(sH2[ri, si, inb])
                out["Sigma_gas_eff"][ri, si] = np.nanmean(sHI[ri, si, inb] + sH2[ri, si, inb])
    return out

In [ ]:
# ── §4 DIAGS + §7 plotters + §8g clock engine (operate on bound globals; see Part 3) ──
def build_diags(records, P, cols, STAGES):
    """diags[stage][quantity] -> per-record array (NaN if missing). Catalogue diagnostics."""
    def gather(prop, st):
        rows = np.array([r[f"row_{st}"] for r in records]); out = np.full(len(records), np.nan)
        arr = P.get(prop)
        if arr is None:
            return out
        for i, (row, col) in enumerate(zip(rows, cols)):
            if row >= 0:
                out[i] = arr[row, col]
        return out
    def ratio(num, den, st):
        a, b = gather(num, st), gather(den, st)
        with np.errstate(all="ignore"):
            return np.where(b > 0, a / b, np.nan)
    D = {}
    for st in STAGES:
        D[st] = {
            "Mdust": gather("masses.dust", st), "MH2": gather("masses.H2", st),
            "MHI": gather("masses.HI", st), "Mstar": gather("masses.stellar", st),
            "Mgas": gather("masses.gas", st), "age": gather("ages.mass_weighted", st),
            "sSFR": ratio("sfr", "masses.stellar", st),
            "dust/M*": ratio("masses.dust", "masses.stellar", st),
            "H2/M*": ratio("masses.H2", "masses.stellar", st),
            "HI/M*": ratio("masses.HI", "masses.stellar", st),
            "dust/gas": ratio("masses.dust", "masses.gas", st),
            "Rgas/Rstar": ratio("radii.gas_half_mass", "radii.stellar_half_mass", st),
            "C20/80*": ratio("radii.stellar_r20", "radii.stellar_r80", st),
            "C20/80gas": ratio("radii.gas_r20", "radii.gas_r80", st)}
    return D

# These plot engines read module globals (records, STAGES, P, t_cosmic_yr, is_fast, is_slow,
# mbin, NBINS, MASS_BIN_LABELS, COL_A/COL_B/LBL_A/LBL_B, DIAGS, DP*) — bound per anchor by
# activate_anchor() in Part 3.
def violin_stage_g(diags, qty, binarr, binlabels, binname="logM*", logy=True, ylabel=None, fname=None):
    nbn = len(binlabels)
    fig, axs = plt.subplots(1, nbn, figsize=(5.0 * nbn, 4.5), sharey=True); axs = np.atleast_1d(axs)
    for bi in range(nbn):
        ax = axs[bi]; binmask = binarr == bi
        for j, st in enumerate(STAGES):
            for off, msk, c in [(-0.18, is_fast & binmask, COL_A), (0.18, is_slow & binmask, COL_B)]:
                v = diags[st][qty][msk]
                v = v[np.isfinite(v) & (v > 0)] if logy else v[np.isfinite(v)]
                if len(v) < 3:
                    continue
                v = np.log10(v) if logy else v
                parts = ax.violinplot([v], positions=[j + off], widths=0.32, showmedians=True)
                for b in parts["bodies"]:
                    b.set_facecolor(c); b.set_alpha(0.6)
                for key in ("cmedians", "cbars", "cmins", "cmaxes"):
                    if key in parts:
                        parts[key].set_color(c)
        ax.set_xticks(range(len(STAGES))); ax.set_xticklabels(STAGES, rotation=35, ha="right")
        ax.set_title(f"{binname} {binlabels[bi]} (n={binmask.sum()})")
    axs[0].set_ylabel(ylabel or (f"log10 {qty}" if logy else qty))
    fig.suptitle(f"{qty} across stages  ({LBL_A} vs {LBL_B})", y=1.02); plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

# ── §8c/§8e event-stack engine (the §8 workhorse): property tracks aligned on triggers ──
# build_registry(B) constructs the per-anchor PROP_REGISTRY + TRIGGERS (fn closures are
# self-contained — they capture that anchor's arrays/records/_ord). _estack/event_stack read
# the globals bound by activate_anchor (t_inc, PROP_REGISTRY, TRIGGERS, is_fast/is_slow, GYR...).
def build_registry(B):
    P = B["P"]; BH = B["BH"]; records = B["records"]; t_ = B["t_cosmic_yr"]; n_snap = B["n_snap"]
    CO = B["CO"]; pr = B["PR"]; SIG = B["SIG"]
    _ord = np.argsort(t_); t_inc = t_[_ord]
    with np.errstate(all="ignore"):
        ssfr_hist = np.where(P["masses.stellar"] > 0, P["sfr"] / P["masses.stellar"], np.nan)
    REG = {}
    def _track(a, col):
        return a[_ord, col].astype(float)
    def _ratio(a, b):
        A, Bd = P.get(a), P.get(b)
        if A is None or Bd is None:
            return None
        with np.errstate(all="ignore"):
            return np.where(Bd > 0, A / np.where(Bd > 0, Bd, np.nan), np.nan)
    def reg(name, arr, logy, label, by="gal"):
        if arr is None:
            return
        fn = (lambda i, a=arr: _track(a, records[i]["col"])) if by == "gal" else (lambda i, a=arr: _track(a, i))
        REG[name] = dict(fn=fn, logy=logy, label=label)
    def reg_stage(name, A, SL, logy, label):
        if A is None or SL is None:
            return
        def fn(i, A=A, SL=SL):
            full = np.full(n_snap, np.nan)
            for si, st in enumerate(SL):
                if si >= A.shape[1]:
                    break
                row = records[i].get(f"row_{st}", -1)
                if isinstance(row, (int, np.integer)) and row >= 0:
                    full[row] = A[i, si]
            return full[_ord]
        REG[name] = dict(fn=fn, logy=logy, label=label, sparse=True)
    if BH is not None:
        reg("BHAR", BH["bh_mdot"], True, r"$\dot M_{\rm BH}$"); reg("f_Edd", BH["bh_fedd"], True, r"$f_{\rm Edd}$")
        reg("M_BH", BH["bh_mass"], True, r"$M_{\rm BH}$")
        reg("jet_strength", CO.get("wjet_hist"), False, r"$w_{\rm jet}$")
        reg("xcoup", CO.get("xcoup_hist"), False, r"$x_{\rm coup}$")
        reg("f_gas", CO.get("fgas_hist"), False, r"$f_{\rm gas}$")
    else:
        reg("f_gas", _ratio("masses.gas", "masses.stellar"), False, r"$f_{\rm gas}$")
    reg("sSFR", ssfr_hist, True, "sSFR"); reg("SFR", P.get("sfr"), True, "SFR")
    reg("M_halo", P.get("masses.total"), True, r"$M_{\rm halo}$"); reg("M_star", P.get("masses.stellar"), True, r"$M_*$")
    reg("M_gas", P.get("masses.gas"), True, r"$M_{\rm gas}$"); reg("M_dust", P.get("masses.dust"), True, r"$M_{\rm dust}$")
    reg("M_H2", P.get("masses.H2"), True, r"$M_{\rm H_2}$"); reg("M_HI", P.get("masses.HI"), True, r"$M_{\rm HI}$")
    reg("dust/M*", _ratio("masses.dust", "masses.stellar"), True, r"$M_{\rm dust}/M_*$")
    reg("H2/M*", _ratio("masses.H2", "masses.stellar"), True, r"$M_{\rm H_2}/M_*$")
    reg("HI/M*", _ratio("masses.HI", "masses.stellar"), True, r"$M_{\rm HI}/M_*$")
    reg("dust/gas", _ratio("masses.dust", "masses.gas"), True, r"$M_{\rm dust}/M_{\rm gas}$")
    reg("Rgas/Rstar", _ratio("radii.gas_half_mass", "radii.stellar_half_mass"), False, r"$R_{\rm gas}/R_*$")
    reg("R20g/R80g", _ratio("radii.gas_r20", "radii.gas_r80"), True, r"$R_{20}/R_{80}$ gas")
    reg("R20s/R80s", _ratio("radii.stellar_r20", "radii.stellar_r80"), True, r"$R_{20}/R_{80}$ star")
    reg("Age_s", P.get("ages.mass_weighted"), True, r"Age$_\star$")
    if pr["DP_OK"]:
        SL = pr["DP_STAGES"]
        reg_stage("Rdust/Rstar", pr["DP_RATIO"], SL, False, r"$R_{50}^{\rm dust}/R_{50}^{\star}$")
        if SIG:
            reg_stage("Sigma_H2", SIG.get("Sigma_H2_eff"), SL, True, r"$\Sigma_{\rm H_2}$")
            reg_stage("Sigma_HI", SIG.get("Sigma_HI_eff"), SL, True, r"$\Sigma_{\rm HI}$")
            reg_stage("Sigma_gas", SIG.get("Sigma_gas_eff"), SL, True, r"$\Sigma_{\rm gas}$")
            reg_stage("Sigma_dust", SIG.get("Sigma_dust_eff"), SL, True, r"$\Sigma_{\rm dust}$")
    def _ta(key):
        return np.array([r.get(key, np.nan) for r in records], float)
    TRIG = {"t_AGN": np.asarray(B["t_agn"], float), "t_QT": _ta("t_qt"), "t_SFT": _ta("t_sft"),
            "t_SFpeak": _ta("t_sf_peak"), "t_PostQuench": _ta("t_post_quench"),
            "post_quench": _ta("t_post_quench"), "t_GasMin": _ta("t_gas_min"),
            "gas_min": _ta("t_gas_min"), "t_formation": _ta("t_formation")}
    return dict(_ord=_ord, t_inc=t_inc, ssfr_hist=ssfr_hist, PROP_REGISTRY=REG, TRIGGERS=TRIG)

def _estack(ax, zero_t, arr_of, ylab, logy=True, xwin=2.0, nb=12, band="iqr", ref0=False, split=None):
    """Median (+IQR band) of arr_of(i) around per-record zero_t, per group (default fast/slow).
    ref0=True subtracts each galaxy's own value at the trigger (composition-immune). Verbatim §8c."""
    xe = np.linspace(-xwin, xwin, nb + 1); xc = 0.5 * (xe[:-1] + xe[1:])
    groups = split if split is not None else [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]
    for msk, c, lab in groups:
        ii = np.where(msk & np.isfinite(zero_t))[0]
        prof = np.full((len(ii), len(xc)), np.nan)
        for k, i in enumerate(ii):
            tt = (t_inc - zero_t[i]) / GYR; vv = arr_of(i)
            with np.errstate(divide="ignore", invalid="ignore"):
                vv = np.where(vv > 0, np.log10(vv), np.nan) if logy else vv
            for b in range(len(xc)):
                sel = (tt >= xe[b]) & (tt < xe[b + 1]) & np.isfinite(vv)
                if sel.any():
                    prof[k, b] = np.nanmedian(vv[sel])
        if not len(ii):
            continue
        if ref0:
            b0 = int(np.clip(np.searchsorted(xe, 0.0) - 1, 0, len(xc) - 1))
            for k in range(prof.shape[0]):
                row = prof[k]; fin = np.where(np.isfinite(row))[0]
                if not len(fin):
                    continue
                bref = b0 if np.isfinite(row[b0]) else fin[np.argmin(np.abs(fin - b0))]
                prof[k] = row - row[bref]
        with np.errstate(all="ignore"):
            med = np.nanmedian(prof, axis=0); nbin = np.sum(np.isfinite(prof), axis=0)
            if band == "iqr":
                lo = np.nanpercentile(prof, 25, axis=0); hi = np.nanpercentile(prof, 75, axis=0)
            else:
                lo = hi = None
        med = np.where(nbin >= 3, med, np.nan)
        ax.plot(xc, med, "-o", color=c, label=f"{lab} (n={len(ii)})")
        if lo is not None:
            ax.fill_between(xc, np.where(nbin >= 3, lo, np.nan), np.where(nbin >= 3, hi, np.nan),
                            color=c, alpha=0.15, lw=0)
    ax.axvline(0, ls="--", c="k"); ax.set_ylabel(ylab); ax.legend(fontsize=8)

def event_stack(props, triggers=("t_AGN",), ref0=False, band="iqr", xwin=2.0, nb=12, fname=None, split=None):
    """Event-aligned stacks for any registered property × trigger, comparing groups (default
    fast/slow). props/triggers: name or list. ref0=True -> per-galaxy ΔX. Verbatim §8e."""
    props = [props] if isinstance(props, str) else list(props)
    triggers = [triggers] if isinstance(triggers, str) else list(triggers)
    props = [p for p in props if p in PROP_REGISTRY] or list(PROP_REGISTRY)[:1]
    triggers = [t for t in triggers if t in TRIGGERS] or ["t_QT"]
    nr, nc = len(props), len(triggers)
    fig, axs = plt.subplots(nr, nc, figsize=(5.6 * nc, 3.5 * nr), squeeze=False, sharex="col", sharey="row")
    for ri, pn in enumerate(props):
        spec = PROP_REGISTRY[pn]
        ylab = (r"$\Delta$" if ref0 else "med ") + ("log " if spec["logy"] else "") + spec["label"]
        for ci, tn in enumerate(triggers):
            ax = axs[ri][ci]
            _estack(ax, TRIGGERS[tn], spec["fn"], ylab, logy=spec["logy"], xwin=xwin, nb=nb, band=band, ref0=ref0, split=split)
            if ref0:
                ax.axhline(0, ls=":", c="grey")
            if ri == nr - 1:
                ax.set_xlabel(f"t − {tn}  [Gyr]")
            if ri == 0:
                ax.set_title(tn)
    _grp = "  (" + " vs ".join(g[2] for g in split) + ")" if split else f"  ({SPLIT_DESC})"
    fig.suptitle(f"event-aligned {'Δ-' if ref0 else ''}stacks{_grp}, 25–75% bands", y=1.0)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()
    return fig

In [ ]:
# ── §8g·shape full-history SFH/reservoir stacks on a movable critical-point clock ──
_SFH_QTY = {"sfr": ("sfr",), "ssfr": ("sfr", "masses.stellar"),
            "gas/M*": ("masses.gas", "masses.stellar"), "dust/M*": ("masses.dust", "masses.stellar"),
            "H2/M*": ("masses.H2", "masses.stellar"), "HI/M*": ("masses.HI", "masses.stellar"),
            "dust/gas": ("masses.dust", "masses.gas")}
_SFH_YLAB = {"sfr": r"$\log_{10}$ SFR", "ssfr": r"$\log_{10}$ sSFR [1/yr]",
             "gas/M*": r"$\log_{10}\,M_{\rm gas}/M_\star$", "dust/M*": r"$\log_{10}\,M_{\rm dust}/M_\star$",
             "H2/M*": r"$\log_{10}\,M_{\rm H_2}/M_\star$", "HI/M*": r"$\log_{10}\,M_{\rm HI}/M_\star$",
             "dust/gas": r"$\log_{10}\,M_{\rm dust}/M_{\rm gas}$"}

def _sfh_track(col, kind="sfr"):
    spec = _SFH_QTY[kind]; num = P[spec[0]][:, col].astype(float)
    if len(spec) == 1:
        q = num
    else:
        den = P[spec[1]][:, col].astype(float); q = np.where(den > 0, num / den, np.nan)
    good = np.isfinite(q) & (q > 0) & np.isfinite(t_cosmic_yr)
    if good.sum() < 3:
        return None, None
    ts, ys = t_cosmic_yr[good], np.log10(q[good]); o = np.argsort(ts)
    return ts[o], ys[o]

def sfh_shape(anchors=("sf_peak", "qt", "gas_min"), kind="ssfr", split=None,
              xwin=(-12.0, 10.0), nx=61, min_n=5, facet_mass=False, fname=None):
    """Median (+IQR) log-quantity stacked on each critical-point clock, per population."""
    groups = split if split is not None else [(is_fast, COL_A, LBL_A), (is_slow, COL_B, LBL_B)]
    xg = np.linspace(xwin[0], xwin[1], nx); rows = list(range(NBINS)) if facet_mass else [None]
    fig, axes = plt.subplots(len(rows), len(anchors), squeeze=False,
                             figsize=(4.6 * len(anchors), 3.6 * len(rows)), sharex=True, sharey="row")
    for ri, bi in enumerate(rows):
        binmask = np.ones(len(records), bool) if bi is None else (mbin == bi)
        for ci, st in enumerate(anchors):
            ax = axes[ri][ci]; tkey = f"t_{st}"
            for msk, c, lab in groups:
                acc = []
                for i in np.where(msk & binmask)[0]:
                    t0 = records[i].get(tkey, np.nan)
                    if not np.isfinite(t0):
                        continue
                    ts, ys = _sfh_track(records[i]["col"], kind)
                    if ts is None:
                        continue
                    acc.append(np.interp(xg, (ts - t0) / 1e9, ys, left=np.nan, right=np.nan))
                if not acc:
                    continue
                M = np.asarray(acc); nbin = np.sum(np.isfinite(M), axis=0)
                with np.errstate(all="ignore"):
                    med = np.nanmedian(M, axis=0); lo = np.nanpercentile(M, 25, axis=0); hi = np.nanpercentile(M, 75, axis=0)
                keep = nbin >= min_n
                ax.plot(xg, np.where(keep, med, np.nan), "-", color=c, lw=2, label=f"{lab} (n={len(acc)})")
                ax.fill_between(xg, np.where(keep, lo, np.nan), np.where(keep, hi, np.nan), color=c, alpha=0.15, lw=0)
            ax.axvline(0, ls=":", c="k", lw=1)
            if ri == 0:
                ax.set_title(f"aligned on {st}")
            if ri == len(rows) - 1:
                ax.set_xlabel(rf"$t - t_{{\rm {st}}}$ [Gyr]")
            if ci == 0:
                ax.set_ylabel(((f"logM* {MASS_BIN_LABELS[bi]}\n" if bi is not None else "") + "med " + _SFH_YLAB[kind]))
            ax.legend(fontsize=8)
    fig.suptitle(f"critical-point-clock stacks — {kind};  split: " + " vs ".join(g[2] for g in groups), y=1.01)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

def dust_split(thresh=-5.0, stage="gas_min", colors=("C2", "0.5")):
    """Split on log10(M_dust/M*) at `stage` (default gas_min): dusty vs non-dusty."""
    dfr = DIAGS[stage]["dust/M*"]; defi = np.isfinite(dfr)
    dusty = defi & (dfr > 10.0 ** thresh); nond = defi & ~(dfr > 10.0 ** thresh)
    return [(dusty, colors[0], f"dusty (log Md/M*>{thresh:g} @{stage})"), (nond, colors[1], "non-dusty")]

# Part 3 — Driver: build every anchor's products

Pure load + in-memory analysis (records, QC, AGN metrics, coupling, DIAGS, profiles). Heavy
products are loaded from cache; missing ones degrade to NaN with a warning. `RESULTS[z]` bundles
everything; `activate_anchor(z)` binds the engine globals to that anchor.

In [ ]:
RESULTS = {}
for _zt, A in ANCHORS.items():
    if not os.path.exists(A["hist_path"]):
        print(f"[{A['tag']}] history missing -> set BUILD_MULTI_Z=True (cluster). Skipping."); continue
    H = load_anchor_history(A)
    P_ = H["P"]; t_ = H["t_cosmic_yr"]; z_ = H["redshift"]; sn_ = H["snaps_arr"]; gid_ = H["galaxy_ids"]
    n_snap, n_gal = P_["sfr"].shape
    smask, _ = passive_sample_mask(P_, t_)
    scols = np.where(smask)[0]
    REC = build_records(P_, t_, z_, gid_, scols, MASS_BIN_EDGES, HMASS_BIN_EDGES,
                        PASSIVE_FACTOR, TAU_SPLIT_LOG, COSMO, find_quenching_times)
    records = REC["records"]; STAGES = REC["STAGES"]; cols = REC["cols"]
    if len(records) == 0:
        print(f"[{A['tag']}] no usable quench events; skipping."); continue
    QC = qc_anchor(P_, t_, z_, sn_, gid_, cols, BOX, f"[{A['tag']}]")
    # AGN history (optional build)
    if BUILD_BH and not os.path.exists(A["bh_path"]):
        build_bh_for_anchor(A, gid_, sn_, n_gal)
    BH = load_bh(A["bh_path"]) if os.path.exists(A["bh_path"]) else None
    if BH is not None:
        t_agn = agn_activation_time(BH, records, t_)
        MET = bh_metrics(BH, records, t_, EPS_R, c2_erg_per_Msun, JET_LOGMBH, JET_FEDD)
        CO = build_coupling(BH, P_, records, t_, t_agn, JET_LOGMBH, JET_FEDD, XRAY_FEDD_MAX, XRAY_FGAS_MAX)
    else:
        print(f"[{A['tag']}] BH history missing -> coupling NaN (set BUILD_BH=True).")
        nan = np.full(len(records), np.nan); z0 = np.zeros(len(records), bool)
        t_agn = nan.copy(); MET = dict(Eagn=nan, dMbh=nan, jetfrac=nan, mdot_qt=nan)
        CO = dict(fgas_hist=None, wjet_hist=None, xray_hist=None, xcoup_hist=None,
                  xfrac_post=nan.copy(), xstr_post=nan.copy(), tercile=(np.nan, np.nan),
                  no_agn=z0.copy(), weak=z0.copy(), strong=z0.copy(),
                  SPLIT_X=[(z0.copy(), "C3", "ISM-coupled"), (z0.copy(), "C0", "never coupled")],
                  SPLIT_XS=[(z0.copy(), "C3", "strong coupling"), (z0.copy(), "C0", "weak coupling")])
    # add AGN activation as a critical point (stage "agn"), inserted after ssfr_min
    if "agn" not in STAGES:
        STAGES.insert(STAGES.index("ssfr_min") + 1, "agn")
    for i, r in enumerate(records):
        r["t_agn"] = t_agn[i]
        r["row_agn"] = int(np.argmin(np.abs(t_ - t_agn[i]))) if np.isfinite(t_agn[i]) else -1
    DIAGS = build_diags(records, P_, cols, STAGES)
    PR = load_profiles(A["prof_path"], records)
    SIG = sigma_scalars(PR["DP"]) if PR["DP_OK"] else {}

    RESULTS[_zt] = dict(
        z=_zt, A=A, n_snap=n_snap, n_gal=n_gal,
        P=P_, t_cosmic_yr=t_, redshift=z_, snaps_arr=sn_, galaxy_ids=gid_,
        sample_cols=scols, records=records, STAGES=STAGES, cols=cols,
        tau_q=REC["tau_q"], tau_q_over_tH=REC["tau_q_over_tH"],
        is_fast=REC["is_fast"], is_slow=REC["is_slow"], mbin=REC["mbin"], hmbin=REC["hmbin"],
        logm_rec=REC["logm_rec"], logmh_rec=REC["logmh_rec"],
        well_behaved=QC["well_behaved"], QC=QC, BH=BH, t_agn=t_agn, MET=MET, CO=CO,
        DIAGS=DIAGS, PR=PR, SIG=SIG)
    RESULTS[_zt]["REG"] = build_registry(RESULTS[_zt])   # per-anchor PROP_REGISTRY + TRIGGERS
    del H; gc.collect()

print("\nanchors built:", {z: len(RESULTS[z]["records"]) for z in RESULTS})

In [ ]:
def activate_anchor(z):
    """Bind the module globals the reused engines read to anchor `z`'s arrays.
    QC is folded into is_fast/is_slow so every plot respects the well-behaved mask."""
    B = RESULTS[z]; g = globals()
    wb = B["well_behaved"]
    ns = dict(
        P=B["P"], t_cosmic_yr=B["t_cosmic_yr"], redshift=B["redshift"], snaps_arr=B["snaps_arr"],
        galaxy_ids=B["galaxy_ids"], records=B["records"], STAGES=B["STAGES"], cols=B["cols"],
        mbin=B["mbin"], hmbin=B["hmbin"], is_fast=B["is_fast"] & wb, is_slow=B["is_slow"] & wb,
        is_fast_raw=B["is_fast"], is_slow_raw=B["is_slow"], well_behaved=wb,
        DIAGS=B["DIAGS"], BH=B["BH"], t_agn=B["t_agn"], CO=B["CO"], MET=B["MET"])
    pr = B["PR"]
    ns.update(DP=pr["DP"], DP_OK=pr["DP_OK"], DP_STAGES=pr["DP_STAGES"], DP_RMID=pr["DP_RMID"],
              DP_COMPS=pr["DP_COMPS"], DP_RATIO=pr["DP_RATIO"], DP_C2050=pr["DP_C2050"],
              DP_C2080=pr["DP_C2080"], DP_HAS_CONC=pr["DP_HAS_CONC"], SIG=B["SIG"])
    reg = B["REG"]                                   # event-stack engine globals
    ns.update(_ord=reg["_ord"], t_inc=reg["t_inc"], ssfr_hist=reg["ssfr_hist"],
              PROP_REGISTRY=reg["PROP_REGISTRY"], TRIGGERS=reg["TRIGGERS"],
              SPLIT_X=B["CO"]["SPLIT_X"], SPLIT_XS=B["CO"]["SPLIT_XS"], SPLIT_DESC="fast vs slow")
    g.update(ns)
    return B

print("activate_anchor ready. anchors:", list(RESULTS))

In [ ]:
# ── BUILD_PROFILE_PLAN (GATED): write per-anchor SLURM plans, then run on the cluster ──
# Each plan lives in its OWN dir (output/cis100/caesar_sfh/prof_<tag>/) so the job's
# profiles_part_*.hdf5 partials never collide across anchors. Run ONCE PER ANCHOR:
#   PLAN=output/cis100/caesar_sfh/prof_<tag>/dust_profile_plan_<tag>.hdf5
#   DUST_PLAN=$PLAN sbatch --array=0-N submit_profiles.sh        # build_profiles_job.py
#   DUST_PLAN=$PLAN \
#     DUST_FINAL=output/cis100/caesar_sfh/dust_profiles_allcrit_<tag>.hdf5 \
#     CGMPROF_FINAL=output/cis100/caesar_sfh/cgm_profiles_allcrit_<tag>.hdf5 \
#     CGMT_FINAL=output/cis100/caesar_sfh/cgm_temperature_allcrit_<tag>.hdf5 \
#     python merge_profiles.py
# (merge_profiles.py already routes per-anchor outputs via these env vars — no repo edit needed.
#  DUST_OUTDIR defaults to the plan's dir, so partials & merge stay inside prof_<tag>/.)
if BUILD_PROFILE_PLAN:
    for _zt, B in RESULTS.items():
        A = B["A"]
        PIDX = build_prog_index(A, B["galaxy_ids"], B["snaps_arr"])
        build_profile_plan(B["records"], B["STAGES"], B["snaps_arr"], PIDX,
                           A["plan_path"], SIM_NAME)
else:
    print("BUILD_PROFILE_PLAN=False -> expecting cached per-anchor profile products.")
    for _zt, B in RESULTS.items():
        print(f"  [{B['A']['tag']}] profiles:", "ok" if B["PR"]["DP_OK"] else "MISSING")

# Part 4 — QC summary across anchors

Before any science: how clean are the samples? Broken progenitor links, halo-mass teleports and
bad halo associations are tabulated per anchor; all downstream analysis uses only `well_behaved`
galaxies (folded into the fast/slow masks by `activate_anchor`).

In [ ]:
print(f"{'z':>5s} {'N_rec':>6s} {'well':>5s} {'broken':>7s} {'haloJmp':>8s} "
      f"{'badAssoc':>9s} {'noHalo':>7s} {'medSHMR':>8s}")
for z in RESULTS:
    B = RESULTS[z]; Q = B["QC"]; n = len(B["records"])
    print(f"{z:5.1f} {n:6d} {int(Q['well_behaved'].sum()):5d} {Q['n_broken_link']:7d} "
          f"{Q['n_halo_teleport']:8d} {Q['n_bad_assoc']:9d} {Q['n_no_halo']:7d} {Q['median_shmr']:8.4f}")
print("\nNOTE: spatial-continuity uses P['pos'] with auto comoving/physical detection from the "
      "tracked extent; if a column over-flags, inspect that anchor's positions before trusting it.")

# Part 5 — Sample composition

Fast/slow and AGN-coupling subsample counts per anchor and mass bin — the populations every
comparison below is built on.

In [ ]:
print(f"{'z':>5s} {'mass bin':>10s} {'N':>4s} {'fast':>5s} {'slow':>5s} "
      f"{'no-AGN':>7s} {'weak':>5s} {'strong':>7s}")
for z in RESULTS:
    B = activate_anchor(z); CO = B["CO"]; wb = B["well_behaved"]
    for bi, lab in enumerate(MASS_BIN_LABELS):
        m = (mbin == bi) & wb
        print(f"{z:5.1f} {lab:>10s} {int(m.sum()):4d} {int((m & is_fast).sum()):5d} "
              f"{int((m & is_slow).sum()):5d} {int((m & CO['no_agn']).sum()):7d} "
              f"{int((m & CO['weak']).sum()):5d} {int((m & CO['strong']).sum()):7d}")

In [ ]:
# ── pooled record-level scalars across anchors (well-behaved). gas_min is used ONLY to flag
#    dusty vs non-dusty; everything else is event-aligned (Part 6). ──
def _sig_at(B, key, stage):
    SIG = B["SIG"]; DPS = B["PR"]["DP_STAGES"]
    if not SIG or DPS is None or stage not in DPS:
        return np.full(len(B["records"]), np.nan)
    return SIG[key][:, DPS.index(stage)]

def _diag_at(B, key, stage):
    return B["DIAGS"][stage][key] if stage in B["DIAGS"] else np.full(len(B["records"]), np.nan)

def _wb_split(split):
    """Fold the active anchor's well_behaved mask into a [(mask,color,label),...] split spec."""
    return [(m & well_behaved, c, l) for (m, c, l) in split]

POOL = {k: [] for k in ("z", "is_fast", "is_slow", "mbin", "tau_q_over_tH", "xstr_post",
                        "no_agn", "weak", "strong", "lag_qt_agn",
                        "SigH2_qt", "SigH2_postq", "dust_postq", "dustfrac_gasmin")}
for z in RESULTS:
    B = RESULTS[z]; wb = B["well_behaved"]; n = len(B["records"])
    lag = (np.array([r["t_qt"] for r in B["records"]]) - np.asarray(B["t_agn"], float)) / GYR
    cmap = {
        "z": np.full(n, z), "is_fast": B["is_fast"], "is_slow": B["is_slow"], "mbin": B["mbin"],
        "tau_q_over_tH": B["tau_q_over_tH"], "xstr_post": B["CO"]["xstr_post"],
        "no_agn": B["CO"]["no_agn"], "weak": B["CO"]["weak"], "strong": B["CO"]["strong"],
        "lag_qt_agn": lag,
        "SigH2_qt": _sig_at(B, "Sigma_H2_eff", "qt"),
        "SigH2_postq": _sig_at(B, "Sigma_H2_eff", "post_quench"),
        "dust_postq": _diag_at(B, "dust/M*", "post_quench"),
        "dustfrac_gasmin": _diag_at(B, "dust/M*", "gas_min")}
    for k, v in cmap.items():
        POOL[k].append(np.asarray(v)[wb])
POOL = {k: (np.concatenate(v) if v else np.array([])) for k, v in POOL.items()}
DUST_THR = -3.0   # log10(M_dust/M*) at gas_min separating dusty/non-dusty (tune per sample)
POOL["dusty"] = np.isfinite(POOL["dustfrac_gasmin"]) & (POOL["dustfrac_gasmin"] > 10.0 ** DUST_THR)
print("pooled well-behaved records:", len(POOL["z"]),
      f"| dusty-at-gas_min (log Md/M*>{DUST_THR}): {int(POOL['dusty'].sum())}")

# Part 6 — Event-aligned analysis across redshift (§8-style)

The core comparison: property *tracks* aligned on the critical points (`t_SFpeak`, `t_AGN`, `t_SFT`,
`t_QT`, `post_quench`) over a ±window, comparing galaxies at different times **after quenching**.
The organizing axis is **AGN–ISM coupling** (`SPLIT_XS` = terciles of post-`t_AGN` coupling strength,
the §8j definition). `gas_min` enters only as the dusty/non-dusty divider (6e). Each cell loops over
the anchors; coupling/dusty splits are folded through `well_behaved`.

### 6a — Coupling & dusty/non-dusty splits (counts per anchor)

In [ ]:
print(f"{'z':>5s} {'N_wb':>5s} {'strong':>7s} {'weak':>5s} {'no-AGN':>7s} "
      f"{'ISMcoup':>8s} {'dusty@gmin':>11s}")
for z in RESULTS:
    B = activate_anchor(z); CO = B["CO"]; wb = well_behaved
    nstrong = int((CO["strong"] & wb).sum()); nweak = int((CO["weak"] & wb).sum())
    nno = int((CO["no_agn"] & wb).sum())
    ncoup = int((CO["SPLIT_X"][0][0] & wb).sum())
    df = _diag_at(B, "dust/M*", "gas_min")
    ndusty = int((np.isfinite(df) & (df > 10.0 ** DUST_THR) & wb).sum())
    print(f"{z:5.1f} {int(wb.sum()):5d} {nstrong:7d} {nweak:5d} {nno:7d} {ncoup:8d} {ndusty:11d}")
print(f"\ncoupling terciles per anchor: tercile(lo,hi)=",
      {z: tuple(np.round(RESULTS[z]['CO']['tercile'], 3)) for z in RESULTS})

### 6a·clock — Coupling strength around each critical point (fast vs slow)

The continuous AGN–ISM coupling `x_coup` (jet strength gated by gas-poorness) stacked on each
critical-point clock, **fast vs slow**. A direct read of *when* the coupling switches on relative to
the SF peak, AGN ignition, SFT, QT, post-quench and gas_min — and whether fast quenchers couple
earlier or harder. Needs the BH history (coupling).

In [ ]:
# surface-density tracks added to every Part-6 stack (registered only when the profile product exists)
SIGMA_PROPS = ["Sigma_H2", "Sigma_HI", "Sigma_gas", "Sigma_dust"]
COUP_TRIG = ["t_SFpeak", "t_AGN", "t_SFT", "t_QT", "post_quench", "gas_min"]
for z in RESULTS:
    B = activate_anchor(z)   # default split = is_fast / is_slow (well_behaved-folded)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling unavailable (set BUILD_BH=True)."); continue
    print(f"\n=== z = {z} : x_coup + ISM surface densities around the critical points (fast vs slow) ===")
    event_stack([p for p in ["xcoup"] + SIGMA_PROPS if p in PROP_REGISTRY], triggers=COUP_TRIG, xwin=2.0, nb=10,
                fname=f"coupling_clock_fastslow_z{_ztag(z)}.png")

### 6b — The scenario stack, by coupling strength (the canonical §8j call)

`dust/M*`, `H2/M*`, gas & stellar compactness (`R20g/R80g`, `R20s/R80s`) and `Rdust/Rstar` aligned on
each critical point, **strong vs weak coupling**. The strong-coupling prediction: earlier/steeper
compaction and a dust rise after `t_QT`. Needs BH history (coupling); profile product adds `Rdust/Rstar`.

In [ ]:
SCEN = ["dust/M*", "H2/M*", "R20g/R80g", "R20s/R80s", "Rdust/Rstar"] + SIGMA_PROPS
SCEN_TRIG = ["t_SFpeak", "t_AGN", "t_SFT", "t_QT", "post_quench"]
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling split unavailable (set BUILD_BH=True)."); continue
    print(f"\n=== z = {z} : scenario stack by coupling strength ===")
    event_stack([p for p in SCEN if p in PROP_REGISTRY], triggers=SCEN_TRIG,
                split=_wb_split(SPLIT_XS), xwin=2.0, nb=10, fname=f"scenario_xstrength_z{_ztag(z)}.png")

### 6c — The same stack, organized by fast/slow (compare the two axes)

Identical properties/triggers split by quench *time* instead of coupling. If coupling (6b) separates
the tracks more cleanly than fast/slow does here, quench time is the weaker organizing axis.

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)   # default split = is_fast/is_slow (already well_behaved-folded)
    print(f"\n=== z = {z} : scenario stack by fast/slow ===")
    event_stack([p for p in SCEN if p in PROP_REGISTRY], triggers=SCEN_TRIG,
                xwin=2.0, nb=10, fname=f"scenario_fastslow_z{_ztag(z)}.png")

### 6d — SF & AGN response around ignition and quenching

sSFR, BHAR, molecular gas and dust around `t_AGN`, `t_QT`, `post_quench`, by coupling. Shows the
post-quench sSFR behaviour (the rebound) in its proper event-aligned form.

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        continue
    print(f"\n=== z = {z} : SF/AGN response ===")
    event_stack([p for p in ["sSFR", "BHAR", "M_H2", "dust/M*"] + SIGMA_PROPS if p in PROP_REGISTRY],
                triggers=["t_AGN", "t_QT", "post_quench"], split=_wb_split(SPLIT_XS),
                xwin=3.0, nb=12, fname=f"response_xstrength_z{_ztag(z)}.png")

### 6e — Dusty vs non-dusty at gas_min → quenching mode (the end goal)

gas_min's one job: split dusty/non-dusty. Do the dusty-at-gas_min galaxies show the strong-coupling
signature (compact, dust rising after QT)? Event stacks split by dustiness, plus a Fisher test of
dusty-at-gas_min against coupling and fast/slow (pooled).

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)
    ds = _wb_split(dust_split(thresh=DUST_THR, stage="gas_min"))
    print(f"\n=== z = {z} : stacks split by dust-at-gas_min ===")
    event_stack([p for p in ["dust/M*", "H2/M*", "R20g/R80g", "sSFR"] + SIGMA_PROPS if p in PROP_REGISTRY],
                triggers=["t_QT", "post_quench"], split=ds, xwin=3.0, nb=12,
                fname=f"dustsplit_z{_ztag(z)}.png")

def _fisher(a, b):
    a = a.astype(bool); b = b.astype(bool)
    tab = np.array([[np.sum(a & b), np.sum(a & ~b)], [np.sum(~a & b), np.sum(~a & ~b)]])
    if tab.min() == 0 and tab.sum() < 20:
        return tab, np.nan, np.nan
    odds, p = fisher_exact(tab); return tab, odds, p
print("\nFisher: is dusty-at-gas_min associated with...  (pooled, well-behaved)")
for lab, other in [("strong-coupled", POOL["strong"]), ("weak-coupled", POOL["weak"]),
                   ("no-AGN", POOL["no_agn"]), ("fast", POOL["is_fast"]), ("slow", POOL["is_slow"])]:
    tab, odds, p = _fisher(POOL["dusty"], other)
    print(f"  vs {lab:>15s}: odds={odds:.2f}  p={p:.2g}   "
          f"[dusty&{lab}={tab[0,0]}, dusty&not={tab[0,1]}]")

### 6f — Long after quenching: composition-immune ΔX (ref0), strong vs weak

`ref0=True` subtracts each galaxy's value at the trigger, so the curve is the *change* relative to
QT — immune to the changing-cohort offset. The weak-coupling prediction: more persistent residual
dust (shallower decline) and slower compaction at large `t − t_QT`.

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        continue
    print(f"\n=== z = {z} : Δ(after QT), strong vs weak ===")
    event_stack([p for p in ["dust/M*", "Rdust/Rstar", "R20g/R80g"] + SIGMA_PROPS if p in PROP_REGISTRY],
                triggers=["t_QT", "post_quench"], split=_wb_split(SPLIT_XS), ref0=True,
                xwin=4.0, nb=12, fname=f"postqt_delta_z{_ztag(z)}.png")

### 6g — Is gas surface density a better tracer than quenching time? (falsifiable head-to-head)

Pooled: Spearman |r| (vs continuous coupling `xstr_post`) and AUC (separating fast/slow) for
`τ_q` vs `Σ_H2` measured at QT / post_quench. Plus event-aligned Σ tracks by coupling. Σ beats τ_q
only if it shows larger |r| *and* larger |AUC−0.5|.

In [ ]:
def _spearman(a, b):
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 8:
        return np.nan, np.nan, int(m.sum())
    r, p = spearmanr(a[m], b[m]); return r, p, int(m.sum())

def _auc(feature, pos, neg):
    a = feature[pos.astype(bool)]; a = a[np.isfinite(a)]
    b = feature[neg.astype(bool)]; b = b[np.isfinite(b)]
    if len(a) < 5 or len(b) < 5:
        return np.nan
    U, _ = mannwhitneyu(a, b, alternative="two-sided"); return U / (len(a) * len(b))

feat = {"log tau_q/t_H": np.log10(np.where(POOL["tau_q_over_tH"] > 0, POOL["tau_q_over_tH"], np.nan)),
        "log Sigma_H2 @QT": np.log10(np.where(POOL["SigH2_qt"] > 0, POOL["SigH2_qt"], np.nan)),
        "log Sigma_H2 @postQ": np.log10(np.where(POOL["SigH2_postq"] > 0, POOL["SigH2_postq"], np.nan))}
print(f"{'feature':>22s} {'|r| vs coupling':>16s} {'AUC fast/slow':>14s} {'|AUC-0.5|':>10s}")
for name, f in feat.items():
    r, p, n = _spearman(f, POOL["xstr_post"]); auc = _auc(f, POOL["is_fast"], POOL["is_slow"])
    print(f"{name:>22s} {abs(r):16.3f} {auc if np.isfinite(auc) else float('nan'):14.3f} "
          f"{abs(auc - 0.5) if np.isfinite(auc) else float('nan'):10.3f}   (r p={p:.1g}, n={n})")
print("Claim supported if Sigma_H2 shows larger |r| AND larger |AUC-0.5| than tau_q.")
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None or not DP_OK:
        continue
    event_stack([p for p in SIGMA_PROPS if p in PROP_REGISTRY],
                triggers=["t_QT", "post_quench"], split=_wb_split(SPLIT_XS), xwin=3.0, nb=10,
                fname=f"sigma_xstrength_z{_ztag(z)}.png")

### 6h — Full SFH per sample + cross-anchor lead-lag synthesis

Full star-formation histories on the critical-point clock by coupling, and the pooled causal
ordering `t_QT − t_AGN` (does AGN ignition precede quenching, more so for strong coupling?).

In [ ]:
for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        continue
    sfh_shape(anchors=("sf_peak", "qt", "post_quench"), kind="ssfr",
              split=_wb_split(SPLIT_XS), fname=f"sfh_ssfr_xstrength_z{_ztag(z)}.png")

fig, ax = plt.subplots(figsize=(8, 5))
for msk, c, lab in [(POOL["strong"], "C3", "strong"), (POOL["weak"], "C0", "weak"),
                    (POOL["no_agn"], "0.5", "no-AGN")]:
    v = POOL["lag_qt_agn"][msk.astype(bool)]; v = v[np.isfinite(v)]
    if len(v) < 3:
        continue
    ax.hist(v, bins=20, density=True, histtype="step", lw=2, color=c,
            label=f"{lab} (med {np.median(v):+.2f}, n={len(v)})")
ax.axvline(0, ls=":", c="k"); ax.set_xlabel(r"$t_{\rm QT}-t_{\rm AGN}$ [Gyr]  (+ = AGN leads quench)")
ax.set_ylabel("pdf"); ax.legend(); ax.set_title("6h — causal ordering, pooled across anchors")
s = POOL["lag_qt_agn"][POOL["strong"].astype(bool)]; w = POOL["lag_qt_agn"][POOL["weak"].astype(bool)]
s = s[np.isfinite(s)]; w = w[np.isfinite(w)]
if len(s) > 2 and len(w) > 2:
    print(f"lead-lag strong vs weak: MW-p = {mannwhitneyu(s, w).pvalue:.2g} "
          f"(median strong {np.median(s):+.2f}, weak {np.median(w):+.2f} Gyr)")
plt.tight_layout(); plt.savefig(figpath("synthesis_leadlag.png"), dpi=130, bbox_inches="tight"); plt.show()

### 6i — Does strong coupling switch on at different times for different masses?

`x_coup` (jet strength gated by gas-poorness) stacked on each critical-point clock, now **split by
stellar-mass bin** within the **strong-coupled** tercile. If massive galaxies couple *earlier*
(relative to AGN ignition / SFT / QT) than low-mass ones, the activation clock is mass-dependent.
The companion scalar is the per-galaxy coupling-activation lag `t_coup − t_QT` — the first rise of
`x_coup` to half its post-ignition peak — tabulated and plotted by mass bin for **strong vs weak**.


In [ ]:
# ── 6i·a — coupling-activation clock, resolved by stellar-mass bin (strong tercile) ──
MASS_COLORS = ["#2c7bb6", "#fdae61", "#d7191c"]   # one per stellar-mass bin (NBINS)

def mass_split(base=None):
    '''[(mask,color,label),...] splitting a base population into stellar-mass bins (mbin globals).'''
    base = np.ones(len(records), bool) if base is None else np.asarray(base, bool)
    return [(base & (mbin == b), MASS_COLORS[b % len(MASS_COLORS)], f"logM* {MASS_BIN_LABELS[b]}")
            for b in range(NBINS)]

def coupling_activation_time(B, frac=0.5, floor=0.05):
    '''First time x_coup rises to frac*(post-ignition peak), at/after t_AGN (per record, yr).
    Mirrors agn_activation_time but tracks the *coupling* (x_coup), which can lag BHAR ignition
    because coupling additionally needs the disc to become gas-poor.'''
    CO = B["CO"]; recs = B["records"]; t_agn = np.asarray(B["t_agn"], float)
    xc = CO.get("xcoup_hist"); t_coup = np.full(len(recs), np.nan)
    if xc is None:
        return t_coup
    _ord = B["REG"]["_ord"]; t_inc = B["REG"]["t_inc"]
    for i, r in enumerate(recs):
        if not np.isfinite(t_agn[i]):
            continue
        cs = xc[_ord, r["col"]].astype(float); post = t_inc >= t_agn[i]
        m = post & np.isfinite(cs) & (cs > 0)
        if m.sum() < 1:
            continue
        thr = max(frac * np.nanmax(cs[m]), floor)
        cr = np.where(post & np.isfinite(cs) & (cs >= thr))[0]
        if len(cr):
            t_coup[i] = t_inc[cr[0]]
    return t_coup

for z in RESULTS:
    B = activate_anchor(z)
    if B["BH"] is None:
        print(f"[z={z}] no BH history -> coupling unavailable (set BUILD_BH=True)."); continue
    print(f"\n=== z={z} : x_coup activation by stellar-mass bin (strong-coupled tercile) ===")
    event_stack([p for p in ["xcoup"] + SIGMA_PROPS if p in PROP_REGISTRY],
                triggers=COUP_TRIG, split=_wb_split(mass_split(B["CO"]["strong"])),
                xwin=2.0, nb=10, fname=f"coupling_bymass_strong_z{_ztag(z)}.png")

# ── 6i·b — pooled coupling-activation lag (t_coup − t_QT) by mass bin, strong vs weak ──
lag_coup = []
for z in RESULTS:
    B = RESULTS[z]; wb = B["well_behaved"]
    tc = coupling_activation_time(B)
    tq = np.array([r["t_qt"] for r in B["records"]], float)
    lag_coup.append(((tc - tq) / GYR)[wb])
lag_coup = np.concatenate(lag_coup) if lag_coup else np.array([])   # aligned with POOL ordering

print(f"\n{'mass bin':>12s}  {'strong med':>10s} {'n':>4s}  {'weak med':>10s} {'n':>4s}   [t_coup-t_QT, Gyr]")
for b in range(NBINS):
    cells = [MASS_BIN_LABELS[b]]
    for grp in (POOL["strong"], POOL["weak"]):
        v = lag_coup[grp.astype(bool) & (POOL["mbin"] == b)]; v = v[np.isfinite(v)]
        cells += [f"{np.median(v):.2f}" if len(v) else "--", str(len(v))]
    print(f"{cells[0]:>12s}  {cells[1]:>10s} {cells[2]:>4s}  {cells[3]:>10s} {cells[4]:>4s}")

fig, ax = plt.subplots(figsize=(7, 4.5))
for grp, c, lab, dx in [(POOL["strong"], "C3", "strong", -0.1), (POOL["weak"], "C0", "weak", 0.1)]:
    xs, meds, los, his = [], [], [], []
    for b in range(NBINS):
        v = lag_coup[grp.astype(bool) & (POOL["mbin"] == b)]; v = v[np.isfinite(v)]
        if len(v) < 3:
            continue
        xs.append(b + dx); meds.append(np.median(v))
        los.append(np.percentile(v, 25)); his.append(np.percentile(v, 75))
    if xs:
        meds = np.array(meds); los = np.array(los); his = np.array(his)
        ax.errorbar(xs, meds, yerr=[meds - los, his - meds], fmt="o-", color=c, capsize=4, label=lab)
ax.axhline(0, ls=":", c="k"); ax.set_xticks(range(NBINS)); ax.set_xticklabels(MASS_BIN_LABELS)
ax.set_xlabel(r"stellar-mass bin  $\log M_\star$")
ax.set_ylabel(r"$t_{\rm coup}-t_{\rm QT}$ [Gyr]  (median $\pm$ IQR)")
ax.set_title("6i - coupling-activation lag vs mass (pooled across anchors)"); ax.legend()
plt.tight_layout(); plt.savefig(figpath("coupling_lag_bymass.png"), dpi=130, bbox_inches="tight"); plt.show()


### 6j — What resolves the weak-coupling post-quench dust scatter?

The weak-coupling tercile shows the widest spread in residual dust after quenching. Is that scatter
random, or set by a second parameter? Pooled across anchors, the response is `log10(M_dust/M*)` at
`post_quench` (**weak sample only**); the candidate predictors, measured at the same stage, are the
**quenching timescale** `τ_q/t_H`, **stellar mass** `logM*`, **gas fraction** `M_gas/M*`, and
**gas surface density** `Σ_gas`. The strong tercile is shown as a control. Spearman |r| ranks the
single best predictor; a standardized multiple fit (lstsq, statsmodels-free) reports the combined
explained variance in the weak sample.


In [ ]:
# ── pooled predictors for the weak-coupling residual-dust scatter (well-behaved, POOL-aligned) ──
def _pool(fn):
    return (np.concatenate([np.asarray(fn(B), float)[B["well_behaved"]] for B in RESULTS.values()])
            if RESULTS else np.array([]))

mg   = _pool(lambda B: _diag_at(B, "Mgas",  "post_quench"))
ms   = _pool(lambda B: _diag_at(B, "Mstar", "post_quench"))
sgas = _pool(lambda B: _sig_at(B, "Sigma_gas_eff", "post_quench"))
with np.errstate(all="ignore"):
    y_dust = np.log10(np.where(POOL["dust_postq"] > 0, POOL["dust_postq"], np.nan))   # log Md/M* @ post_quench
    fgas   = np.where(ms > 0, mg / ms, np.nan)
    X = {
        r"$\log\,\tau_q/t_H$":          np.log10(np.where(POOL["tau_q_over_tH"] > 0, POOL["tau_q_over_tH"], np.nan)),
        r"$\log M_\star$":              _pool(lambda B: B["logm_rec"]),
        r"$\log\,M_{\rm gas}/M_\star$": np.log10(np.where(fgas > 0, fgas, np.nan)),
        r"$\log\,\Sigma_{\rm gas}$":    np.log10(np.where(sgas > 0, sgas, np.nan)),
    }

weak = POOL["weak"].astype(bool); strong = POOL["strong"].astype(bool)
print(f"weak-sample residual-dust scatter: n={int(np.isfinite(y_dust[weak]).sum())}, "
      f"std(log Md/M*)={np.nanstd(y_dust[weak]):.2f} dex  "
      f"(strong: {np.nanstd(y_dust[strong]):.2f} dex)\n")
print(f"{'predictor':>26s}  {'weak |r|':>9s} {'p':>8s} {'n':>4s}   {'strong |r|':>10s}")
fig, axs = plt.subplots(2, 2, figsize=(11, 9)); axs = axs.ravel()
for ax, (name, xv) in zip(axs, X.items()):
    for msk, c, lab, al in [(strong, "0.7", "strong (control)", 0.35), (weak, "C0", "weak", 0.65)]:
        m = msk & np.isfinite(xv) & np.isfinite(y_dust)
        ax.scatter(xv[m], y_dust[m], s=18, c=c, alpha=al, edgecolor="none", label=f"{lab} (n={int(m.sum())})")
    rw, pw, nw = _spearman(xv[weak], y_dust[weak]); rs, ps, ns = _spearman(xv[strong], y_dust[strong])
    print(f"{name:>26s}  {abs(rw):9.3f} {pw:8.1g} {nw:4d}   {abs(rs):10.3f}")
    ax.set_xlabel(name); ax.set_ylabel(r"$\log_{10}(M_{\rm dust}/M_\star)$ @ post-quench")
    ax.set_title(f"weak |r|={abs(rw):.2f} (p={pw:.1g})"); ax.legend(fontsize=8)
fig.suptitle("6j - resolving the weak-coupling residual-dust scatter (pooled anchors)", y=1.0)
plt.tight_layout(); plt.savefig(figpath("weakdust_predictors.png"), dpi=130, bbox_inches="tight"); plt.show()

# ── standardized multiple fit (lstsq): combined explained variance in the weak sample ──
def _zsc(a):
    return (a - np.nanmean(a)) / (np.nanstd(a) + 1e-12)
keys = list(X)
Xmat = np.column_stack([_zsc(X[k]) for k in keys]); yv = _zsc(y_dust)
m = weak & np.isfinite(yv) & np.all(np.isfinite(Xmat), axis=1)
if m.sum() > len(keys) + 2:
    A = np.column_stack([Xmat[m], np.ones(int(m.sum()))])
    beta, *_ = np.linalg.lstsq(A, yv[m], rcond=None)
    pred = A @ beta
    r2 = 1.0 - np.nansum((yv[m] - pred) ** 2) / np.nansum((yv[m] - np.mean(yv[m])) ** 2)
    print(f"\nstandardized multiple fit (weak, complete rows n={int(m.sum())}): R^2={r2:.2f}")
    for k, bcoef in zip(keys, beta[:-1]):
        print(f"   beta[{k}] = {bcoef:+.2f}")
    print("(|beta| ranks each predictor's unique contribution; R^2 is the variance jointly explained.)")
else:
    print(f"\n[multiple fit] too few complete weak-sample rows (n={int(m.sum())}).")


### 6k — Σ distributions vs stellar age & stellar mass (fast/slow and weak/strong)

Pooled across anchors, the four ISM surface densities (`Σ_H2`, `Σ_HI`, `Σ_gas`, `Σ_dust`, mean inside
R50,*) measured at `post_quench`, scattered against **mass-weighted stellar age** and **stellar mass**,
with a running median per population. Two views: split by quench *time* (fast/slow) and by AGN–ISM
*coupling* (strong/weak). Reads whether the residual ISM column tracks how old / how massive the
remnant is, and whether coupling or timescale better organizes that relation.


In [ ]:
# ── pooled ISM surface densities vs stellar age & mass, by population (well-behaved, all anchors) ──
SIGMA_INFO = [("Sigma_H2_eff", r"$\Sigma_{\rm H_2}$"), ("Sigma_HI_eff", r"$\Sigma_{\rm HI}$"),
              ("Sigma_gas_eff", r"$\Sigma_{\rm gas}$"), ("Sigma_dust_eff", r"$\Sigma_{\rm dust}$")]

def _pool(fn):
    return (np.concatenate([np.asarray(fn(B), float)[B["well_behaved"]] for B in RESULTS.values()])
            if RESULTS else np.array([]))

def _sig_safe(B, key, stage):
    '''_sig_at, but returns NaNs if the Σ key is absent (e.g. RESULTS built before Sigma_HI existed).'''
    SIG = B["SIG"]; DPS = B["PR"]["DP_STAGES"]
    if not SIG or DPS is None or stage not in DPS or key not in SIG:
        return np.full(len(B["records"]), np.nan)
    return SIG[key][:, DPS.index(stage)]

def sigma_distributions(stage="post_quench", split="fastslow", fname=None):
    '''Scatter (+running median) of each Σ at `stage` vs mass-weighted age and logM*, per population.'''
    with np.errstate(all="ignore"):
        age  = _pool(lambda B: _diag_at(B, "age",   stage))
        msv  = _pool(lambda B: _diag_at(B, "Mstar", stage))
        logM = np.log10(np.where(msv > 0, msv, np.nan))
    if split == "fastslow":
        groups = [(_pool(lambda B: B["is_fast"]).astype(bool), "C3", "fast"),
                  (_pool(lambda B: B["is_slow"]).astype(bool), "C0", "slow")]
    else:
        groups = [(_pool(lambda B: B["CO"]["strong"]).astype(bool), "C3", "strong"),
                  (_pool(lambda B: B["CO"]["weak"]).astype(bool),   "C0", "weak")]
    xspecs = [(age, "mass-weighted stellar age [Gyr]"), (logM, r"$\log_{10} M_\star$")]
    fig, axs = plt.subplots(len(SIGMA_INFO), 2, figsize=(11, 3.1 * len(SIGMA_INFO)), sharey="row")
    for ri, (key, lab) in enumerate(SIGMA_INFO):
        with np.errstate(all="ignore"):
            sg = _pool(lambda B, k=key: _sig_safe(B, k, stage))
            y = np.log10(np.where(sg > 0, sg, np.nan))
        for ci, (xv, xlab) in enumerate(xspecs):
            ax = axs[ri][ci]
            for msk, c, glab in groups:
                m = msk & np.isfinite(xv) & np.isfinite(y)
                ax.scatter(xv[m], y[m], s=14, c=c, alpha=0.4, edgecolor="none", label=f"{glab} (n={int(m.sum())})")
                if m.sum() >= 8:
                    xe = np.linspace(np.nanpercentile(xv[m], 2), np.nanpercentile(xv[m], 98), 7)
                    xc = 0.5 * (xe[:-1] + xe[1:]); med = np.full(len(xc), np.nan)
                    for b in range(len(xc)):
                        s = m & (xv >= xe[b]) & (xv < xe[b + 1])
                        if s.sum() >= 3:
                            med[b] = np.nanmedian(y[s])
                    ax.plot(xc, med, "-", color=c, lw=2)
            if ri == len(SIGMA_INFO) - 1:
                ax.set_xlabel(xlab)
            if ci == 0:
                ax.set_ylabel(f"log {lab}")
            if ri == 0:
                ax.legend(fontsize=8)
    fig.suptitle(f"Sigma @ {stage} vs age & mass  ({groups[0][2]} vs {groups[1][2]}, pooled anchors)", y=1.0)
    plt.tight_layout()
    if fname:
        plt.savefig(figpath(fname), dpi=130, bbox_inches="tight")
    plt.show()

for _split in ("fastslow", "weakstrong"):
    sigma_distributions(stage="post_quench", split=_split, fname=f"sigma_dist_{_split}_postq.png")


# Part 7 — Interpretation

*(Fill after the cluster run.)* Read against the strong/weak-coupling scenario:

- **6b vs 6c** — does **coupling** (SPLIT_XS) separate the dust / H2 / compactness tracks around the
  critical points more cleanly than **fast/slow** does? If so, the quenching *mode* (not its
  *timescale*) is the organizing variable.
- **Strong coupling** (X-ray, radiatively efficient): earlier compaction (`R20/R80` rises), a dust
  rise after `t_QT` (6b/6d), and — long after QT — a *steeper* dust decline (6f, ref0) → less
  residual dust at late times.
- **Weak coupling** (jet/maintenance): slower compaction, no post-QT sSFR rise, and more persistent
  residual dust (shallower 6f decline).
- **6e** — are the dustiest-at-gas_min galaxies preferentially strong-coupled (Fisher), and do their
  event stacks match the strong-coupling signature?
- **6g** — the falsifiable Σ-vs-τ_q test, now with Σ measured at QT / post-quench rather than gas_min.
- **6h** — does AGN ignition lead quenching (`t_QT−t_AGN`>0), more so for strong coupling?

Caveats to check first: per-anchor N and coupling tercile edges (6a), QC over-flagging (Part 4),
and the `DUST_THR` dusty/non-dusty cut.